# 17  Gabarit d'un formulaire à partir de son PDF vierge, puis découpe des scans question par question

**Ce que fait ce notebook, de haut en bas, sans rien régler à la main**

**Les autres notebooks** (13 : adresses ; puis profession, date de naissance…) utilisent ce travail de deux façons. Soit ils lisent directement les découpes déposées, grâce à l'index. Soit ils chargent le gabarit depuis `data/decoupes/gabarits/` et découpent ce qu'ils veulent (par variable, libellé, question)




In [ ]:
# 1. REGLAGES : tout ce qu'on peut changer est ici
import base64
import getpass as _gp
import hashlib
import importlib
import io
import json
import os
import re
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import s3fs
from PIL import Image

for module_, paquet_ in [("pdfplumber", "pdfplumber"), ("pypdfium2", "pypdfium2"), ("cv2", "opencv-python-headless"),
                         ("openpyxl", "openpyxl")]:
    try:
        importlib.import_module(module_)
    except ImportError:                     # absent du service : installe une fois
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet_], check=True)

BUCKET = "projet-mesure-qualite-rp"
CAMPAGNE = "RG25"
# ---- ou sont les choses, dans l'espace du projet (stockage S3 du SSP Cloud) ----
PDF_FORMULAIRES = f"{BUCKET}/documentation/Questionnaires"      # les PDF VIERGES (un dossier ou un fichier .pdf). Introuvable :
                                                    # le notebook cherche tous les « *imprime*.pdf » du projet.
SCANS = f"{BUCKET}/data/raw/{CAMPAGNE}"            # les scans
DECOUPES = f"{BUCKET}/data/decoupes"               # OU TOUT EST DEPOSE : gabarits/ (configurations) et <CAMPAGNE>/ (decoupes)
FORMULAIRE = None                   # None = deduit du nom du PDF (« 2025-imprime-3-BI avec VAR.pdf » -> BI)
MILLESIME = None                    # None = l'annee dans le nom du PDF
ORDRE_SCAN = "auto"                 # ordre des pages dans un scan : "auto" (numeros imprimes), "pdf", ou ex. [1, 2, 3, 4]

# ---- la decoupe des scans ----
DECOUPER = {"BI": "BI"}             # formulaire -> fin du nom des images a decouper (6901_53..._BI.jpg)
QUESTIONS = "toutes"                # "toutes" (chaque bloc qui attend une reponse) ou ex. ["q23", "q27", "cadre_nom"]
TEST_RAPIDE = False                  # True d'abord : 20 scans ; puis False : tous (reprise automatique)
N_TEST = 400
FORMAT = "jpg"                      # "jpg" (qualite 92, leger) ou "png" (sans perte, ~3 fois plus lourd)
N_PARALLELE_DECOUPE = 8
DUREE_MAX_MINUTES = 90              # au-dela : arret propre, relancer la cellule reprend la ou on s'est arrete

# ---- les modeles (internes au SSP Cloud : aucune image ne sort) ----
MODELE_GEMMA = "gemma4-26b-moe"
UTILISER_MODELE = True              # gemma4 donne le theme et un intitule court de chaque bloc du gabarit
VERIFIER_PAR_IMAGE = True           # gemma4 regarde chaque bloc du gabarit : question entiere ?
MODELES_DETECTION = "auto"          # modeles qui cherchent eux-memes le cadre d'une question : "auto" = tous les modeles
                                    # de vision internes (gemma, qwen-vl, pixtral, llama...) ; ou une liste de noms
N_SCANS_DETECTION = 5               # scans x QUESTIONS_DETECTION pour comparer les modeles au recalage
QUESTIONS_DETECTION = ["q2", "q13", "q21", "q23", "q27", "q32"]
SECOURS_PAR_MODELE = True           # page non recalee : decoupe par le meilleur modele (si son IoU moyen >= 0,7)
N_VERIF_DECOUPES = 30               # decoupes de vrais scans relues par gemma4 (question entiere ?)
URL_SSP = "https://llm.lab.sspcloud.fr/api/chat/completions"
URL_SSP_MODELES = "https://llm.lab.sspcloud.fr/api/models"
N_PARALLELE = 4
DPI = 200
ENREGISTRER_MLFLOW = True

TRAVAIL = Path.home() / "work" / "banc-htr" / "gabarits"    # copie locale de travail (hors Git)
ILLUS = Path.home() / "work" / "banc-htr" / "illustrations" / "gabarits"   # mosaiques de vrais scans : hors Git
for d_ in [TRAVAIL, ILLUS]:
    d_.mkdir(parents=True, exist_ok=True)


def connexion_s3():
    """Connexion au bucket ; si les identifiants ont expire, demande les nouveaux (Onyxia > Mon compte).
    Si l'erreur revient en cours de seance (403, « Access Key Id ... does not exist », « ExpiredToken ») :
    relancer « fs = connexion_s3() » puis la cellule qui a echoue."""
    for essai in range(2):
        s3fs.S3FileSystem.clear_instance_cache()
        fs_ = s3fs.S3FileSystem()
        try:
            fs_.ls(BUCKET)
            return fs_
        except PermissionError:
            if essai == 1:
                raise RuntimeError("identifiants S3 toujours refuses : redemarrer le service Onyxia, "
                                   "ou recopier les identifiants depuis Onyxia > Mon compte > connexion au stockage")
            print("identifiants S3 expires : copier les nouveaux depuis Onyxia > Mon compte > connexion au stockage")
            for v in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
                os.environ[v] = _gp.getpass(f"{v} : ").strip()


fs = connexion_s3()                 # verifie l'acces au stockage AVANT toute lecture


def lire_vault(chemin):
    adresse, jeton = os.environ.get("VAULT_ADDR"), os.environ.get("VAULT_TOKEN")
    if not adresse or not jeton:
        return {}
    montage, reste = chemin.split("/", 1)
    try:
        r = requests.get(f"{adresse.rstrip('/')}/v1/{montage}/data/{reste}", headers={"X-Vault-Token": jeton}, timeout=30)
        return r.json()["data"]["data"] if r.status_code == 200 else {}
    except Exception:
        return {}


if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = (lire_vault("onyxia-kv/farisazibert/LLM").get("OPENAI_API_KEY")
                                    or _gp.getpass("cle du service LLM du SSP Cloud : ").strip())
if os.environ.get("OPENAI_API_KEY", "").startswith("hvs."):
    raise ValueError("La valeur saisie est un jeton VAULT (hvs.…), pas la cle du service LLM : relancer et coller la bonne cle.")
print(f"acces au stockage : OK\nPDF : {PDF_FORMULAIRES}\nscans : {SCANS}\n"
      f"depot : {DECOUPES}/gabarits (configurations) et {DECOUPES}/{CAMPAGNE}/ (decoupes)\n"
      f"mode {'TEST RAPIDE (' + str(N_TEST) + ' scans)' if TEST_RAPIDE else 'COMPLET'}")

In [ ]:
# 2. LE CODE COMMUN DES GABARITS (construction, selection, recalage, decoupe). Il est ecrit dans un fichier, importe ici,
#    et DEPOSE avec les configurations (data/decoupes/gabarits/gabarits.py) : les autres notebooks le chargent de la.
CODE_GABARITS = r'''"""GABARITS DE FORMULAIRES : construire, a partir du PDF VIERGE d'un formulaire (n'importe quel millesime), une
configuration qui decrit TOUS ses elements ; puis, dans les notebooks d'etude, recaler un scan et decouper ce dont on a
besoin (une question, un champ, plusieurs champs, choisis par numero, variable, libelle ou theme).

Principe : la GEOMETRIE vient du PDF lui-meme (positions exactes des cadres, carres numerotes, cases, champs, textes et
noms de variables) ; le SENS (theme, intitule court, verification visuelle) peut etre ajoute par un modele de langage
(notebook 17). Rien n'est saisi a la main : un nouveau millesime = relancer le notebook 17 sur le nouveau PDF.

Ce fichier ne contient aucune donnee. Dependances : pdfplumber, pypdfium2, numpy, pillow, opencv.
"""
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont

VERSION_SCHEMA = 2


# =====================================================================================================================
# 1. OUTILS
# =====================================================================================================================
def normaliser(t):
    t = unicodedata.normalize("NFKD", str(t or "").strip())
    t = "".join(c for c in t if not unicodedata.combining(c)).upper()
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9 ]", " ", t)).strip()


def slug(t, n=28):
    return re.sub(r"_+", "_", normaliser(t).lower().replace(" ", "_")).strip("_")[:n].strip("_") or "x"


def centre(o):
    return ((o["x0"] + o["x1"]) / 2, (o["top"] + o["bottom"]) / 2)


def dedans(pt, r, tol=0.0):
    return r[0] - tol <= pt[0] <= r[2] + tol and r[1] - tol <= pt[1] <= r[3] + tol


def contient(a, b, tol=0.5):
    """Le rectangle a contient-il le rectangle b ?"""
    return a[0] - tol <= b[0] and a[1] - tol <= b[1] and b[2] <= a[2] + tol and b[3] <= a[3] + tol


def recouvre(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])


def union(rs):
    rs = list(rs)
    return (min(r[0] for r in rs), min(r[1] for r in rs), max(r[2] for r in rs), max(r[3] for r in rs))


def surface(r):
    return max(0.0, r[2] - r[0]) * max(0.0, r[3] - r[1])


def bb(o):
    return (float(o["x0"]), float(o["top"]), float(o["x1"]), float(o["bottom"]))


GRAS = ("Bold", "Black", "Heavy", "Semibold", "SemiBold", "ExtraBold", "Demi")


def est_gras(w):
    return any(g in w["fontname"] for g in GRAS)


def est_italique(w):
    return "Italic" in w["fontname"] or "Oblique" in w["fontname"]


def famille(fontname):
    f = fontname.split("+")[-1]
    return re.split(r"[-,]", f)[0].replace("MT", "")


def lignes_de(mots, tol=2.5, ecart_max=28):
    """Regroupe des mots en lignes (meme hauteur), de haut en bas, chaque ligne de gauche a droite.
    Un grand blanc horizontal (> ecart_max pt) coupe la ligne : deux colonnes ne sont pas melangees."""
    L = []
    for w in sorted(mots, key=lambda w: (-(w["bottom"] - w["top"]), w["top"], w["x0"])):   # les grands mots d'abord
        cy = (w["top"] + w["bottom"]) / 2
        for l in L:
            ref = l[0]
            if ref["top"] - 1 <= cy <= ref["bottom"] + 1 and abs(ref["top"] - w["top"]) < max(tol, 0.6 * (ref["bottom"] - ref["top"])):
                l.append(w)
                break
        else:
            L.append([w])
    L = [sorted(l, key=lambda w: w["x0"]) for l in L]
    coupees = []
    for l in L:
        cour = [l[0]]
        for w in l[1:]:
            if w["x0"] - cour[-1]["x1"] > ecart_max:
                coupees.append(cour)
                cour = [w]
            else:
                cour.append(w)
        coupees.append(cour)
    return sorted(coupees, key=lambda l: (l[0]["top"], l[0]["x0"]))


def texte_de(mots):
    t = " ".join(w["text"] for w in mots)
    t = re.sub(r"(\w)- (\w)", r"\1-\2", t)
    t = re.sub(r"\b(\d+) (er|re|e|ème|eme)\b", r"\1\2", t)
    return re.sub(r"\s+", " ", t).replace(" ,", ",").replace(" .", ".").strip()


# =====================================================================================================================
# 2. LECTURE DU PDF : unites de scan (pages, ou demi-pages pour les feuilles A3 pliees), objets, rendu
# =====================================================================================================================
def ouvrir_pdf(chemin_ou_octets):
    import io as _io
    import pdfplumber
    if isinstance(chemin_ou_octets, (bytes, bytearray)):
        return pdfplumber.open(_io.BytesIO(chemin_ou_octets)), bytes(chemin_ou_octets)
    return pdfplumber.open(str(chemin_ou_octets)), Path(chemin_ou_octets).read_bytes()


def rendre_pages(octets, dpi):
    """Chaque page du PDF en image RGB (pypdfium2 : pas de dependance systeme)."""
    import pypdfium2 as pdfium
    doc = pdfium.PdfDocument(octets)
    out = []
    for i in range(len(doc)):
        pg = doc[i]
        pg.set_cropbox(*pg.get_mediabox())             # meme repere que pdfplumber (certaines pages ont une CropBox decalee)
        out.append(np.array(pg.render(scale=dpi / 72).to_pil().convert("RGB")))
    doc.close()
    return out


def unites_du_pdf(pdf, ordre="auto"):
    """Les UNITES telles qu'elles sont scannees. Une page en hauteur = une unite. Une page en largeur (feuille A3
    pliee : deux pages cote a cote) = deux unites, numerotees par le numero imprime en bas (ex. 4 | 1, 2 | 3).
    ordre : "auto" (par numero imprime s'il existe, sinon ordre du PDF), "pdf" (pages entieres, ordre du PDF),
    ou une liste de numeros imprimes (ex. [1, 2, 3, 4])."""
    unites = []
    for pi, p in enumerate(pdf.pages):
        W, H = float(p.width), float(p.height)
        if ordre != "pdf" and W > 1.25 * H:
            moities = [(0.0, W / 2), (W / 2, W)]
        else:
            moities = [(0.0, W)]
        mots = p.extract_words()
        for (x0, x1) in moities:
            # numero imprime : un nombre seul, tout en bas, pres d'un bord exterieur de la demi-page
            num = None
            for w in mots:
                if w["text"].isdigit() and len(w["text"]) <= 2 and w["top"] > 0.94 * H and x0 <= w["x0"] < x1:
                    if min(w["x0"] - x0, x1 - w["x1"]) < 0.12 * (x1 - x0):
                        num = int(w["text"])
            unites.append({"page_pdf": pi, "x0": x0, "x1": x1, "largeur_pt": x1 - x0, "hauteur_pt": H,
                           "numero_imprime": num})
    if isinstance(ordre, (list, tuple)):
        par_num = {u["numero_imprime"]: u for u in unites}
        unites = [par_num[n] for n in ordre]
    elif ordre == "auto" and len(unites) > len(pdf.pages) and all(u["numero_imprime"] for u in unites):
        unites = sorted(unites, key=lambda u: u["numero_imprime"])
    for i, u in enumerate(unites):
        u["unite"] = i
    return unites


# =====================================================================================================================
# 3. EXTRACTION DES ELEMENTS (une page du PDF)
# =====================================================================================================================
def couleur_centre(img, r, k):
    """Couleur mediane du centre (50 %) d'un petit rectangle : l'interieur d'une case, sans son contour."""
    x0, y0, x1, y1 = r
    dx, dy = (x1 - x0) * 0.25, (y1 - y0) * 0.25
    a = img[int((y0 + dy) * k):max(int((y0 + dy) * k) + 1, int((y1 - dy) * k)),
            int((x0 + dx) * k):max(int((x0 + dx) * k) + 1, int((x1 - dx) * k))]
    return np.median(a.reshape(-1, 3), axis=0) if a.size else np.array([255, 255, 255])


def couleur_interieure(img, r, k):
    """Couleur mediane de l'interieur d'un rectangle (sur le rendu : independant de l'espace colorimetrique du PDF)."""
    x0, y0, x1, y1 = r
    # quatre petites plages pres des coins (a 12-25 % des bords) : le texte au centre (chiffre blanc d'un carre
    # numerote, libelle d'un cadre) ne fausse pas la couleur du fond
    pts = []
    for fx in (0.12, 0.88):
        for fy in (0.12, 0.88):
            cx, cy = x0 + fx * (x1 - x0), y0 + fy * (y1 - y0)
            rx, ry = max(0.6, 0.1 * (x1 - x0)), max(0.6, 0.1 * (y1 - y0))
            a = img[int((cy - ry) * k):int((cy + ry) * k) + 1, int((cx - rx) * k):int((cx + rx) * k) + 1]
            if a.size:
                pts.append(a.reshape(-1, 3))
    if not pts:
        return np.array([255, 255, 255])
    return np.median(np.concatenate(pts), axis=0)


def nature_couleur(c):
    lum = 0.299 * c[0] + 0.587 * c[1] + 0.114 * c[2]
    sat = int(c.max()) - int(c.min())
    if lum > 236 and sat < 22:
        return "clair"
    if lum < 150:
        return "fonce"
    return "teinte"


def variables_imprimees(mots):
    """Les noms de variables imprimes sur les PDF « avec VAR » : une police reservee a eux, dont presque tous les mots
    sont des identifiants en majuscules (SEXE, NUMVOI, NATE_1...). Renvoie (mots variables, autres mots)."""
    motif = re.compile(r"^\(?[A-Z][A-Z0-9_]{1,}\)?$")
    par_fam = {}
    for w in mots:
        par_fam.setdefault(famille(w["fontname"]), []).append(w)
    total = len(mots) or 1
    vfam = {f for f, ws in par_fam.items()
            if len(ws) >= 3 and len(ws) < 0.3 * total and sum(bool(motif.match(w["text"])) for w in ws) >= 0.8 * len(ws)}
    var = [w for w in mots if famille(w["fontname"]) in vfam and motif.match(w["text"])]
    ids = {id(w) for w in var}
    return var, [w for w in mots if id(w) not in ids]


def extraire_page(p, img, k, x_min=0.0, x_max=None):
    """Tous les objets utiles d'une page (ou d'une demi-page [x_min, x_max]) : cadres, marqueurs (carres numerotes),
    cases, cellules a chiffres, champs texte, lignes d'ecriture, mots, variables."""
    x_max = float(p.width) if x_max is None else x_max
    dans_unite = lambda o: x_min - 1 <= (o["x0"] + o["x1"]) / 2 <= x_max + 1
    mots_tous = [w for w in p.extract_words(extra_attrs=["fontname", "size"]) if dans_unite(w)]
    var, mots = variables_imprimees(mots_tous)
    # rectangles (y compris les « courbes » fermees a 4 cotes que certains logiciels produisent)
    rects = []
    for r in list(p.rects) + [c for c in p.curves if len(c.get("pts", [])) in (4, 5)]:
        b0 = bb(r)
        b = (max(b0[0], x_min), b0[1], min(b0[2], x_max), b0[3])     # coupe a la pliure (tableau sur deux pages)
        if b[2] - b[0] < max(3.0, 0.3 * (b0[2] - b0[0])) and not dans_unite(r):
            continue
        w_, h_ = b[2] - b[0], b[3] - b[1]
        if w_ < 3 or h_ < 3 or (w_ * h_ > 0.6 * (x_max - x_min) * float(p.height)):
            continue
        if any(max(abs(b[i] - q["bb"][i]) for i in range(4)) < 1.0 for q in rects):
            continue                                    # meme rectangle (fond + contour) : une seule fois
        rects.append({"bb": b, "couleur": nature_couleur(couleur_interieure(img, b, k)),
                      "centre": nature_couleur(couleur_centre(img, b, k))})
    marqueurs, cases, cellules, textes, cadres = [], [], [], [], []
    for r in rects:
        x0, y0, x1, y1 = r["bb"]
        w_, h_ = x1 - x0, y1 - y0
        carre = abs(w_ - h_) < 0.3 * max(w_, h_)
        dedans_mots = [w for w in mots if dedans(centre(w), r["bb"])]
        court = dedans_mots and len("".join(w["text"] for w in dedans_mots)) <= 3
        sur_fonce = any(q["couleur"] == "fonce" and contient(q["bb"], r["bb"], 0.5) and q is not r for q in rects)
        if court and 6 <= max(w_, h_) <= 24 and min(w_, h_) >= 6 and (
                (r["couleur"] != "clair" and carre and all(est_gras(w) for w in dedans_mots))
                or (r["couleur"] == "fonce" and carre) or (sur_fonce and all(est_gras(w) for w in dedans_mots))):
            marqueurs.append({**r, "texte": "".join(w["text"] for w in dedans_mots), "cote": max(w_, h_)})
        elif r["centre"] == "clair" and carre and max(w_, h_) <= 12.5 and not dedans_mots:
            cases.append(r)
        elif r["centre"] == "clair" and r["couleur"] != "fonce" and 9 <= h_ <= 32 and w_ <= 1.35 * h_:
            cellules.append(r)
        elif r["centre"] == "clair" and r["couleur"] == "clair" and 7 <= h_ <= 36 and w_ > 1.35 * h_:
            textes.append(r)
        elif w_ * h_ >= 1200 and min(w_, h_) >= 12:
            cadres.append(r)
    # marqueurs de question vs petits renvois (« passez en 20 ») : les renvois sont nettement plus petits
    if marqueurs:
        grand = np.percentile([m["cote"] for m in marqueurs], 75)
        for m in marqueurs:
            m["renvoi"] = m["cote"] < 0.82 * grand
    # lignes d'ecriture : traits horizontaux fins, hors bords de rectangles, avec de la place au-dessus pour ecrire
    lignes = []
    tous_champs = cases + cellules + textes
    for l in p.lines:
        if not dans_unite(l) or l["bottom"] - l["top"] > 1 or l["x1"] - l["x0"] < 40 or l.get("linewidth", 1) > 1.2:
            continue
        y = l["top"]
        if any(abs(y - r["bb"][1]) < 1.2 or abs(y - r["bb"][3]) < 1.2 for r in rects
               if r["bb"][0] - 1 <= l["x0"] and l["x1"] <= r["bb"][2] + 1 and r not in cadres):
            continue
        zone = (l["x0"], y - 15, l["x1"], y + 2)
        if any(recouvre(zone, c["bb"]) for c in tous_champs):
            continue
        encre = sum(surface((max(zone[0], w["x0"]), max(zone[1], w["top"]), min(zone[2], w["x1"]), min(zone[3], w["bottom"])))
                    for w in mots if recouvre(zone, bb(w)))
        if encre > 0.3 * surface(zone):
            continue
        lignes.append({"bb": zone, "couleur": "clair", "trait": (l["x0"], y, l["x1"])})
    return {"mots": mots, "variables": var, "cadres": cadres, "marqueurs": marqueurs, "cases": cases,
            "cellules": cellules, "textes": textes, "lignes": lignes}


# =====================================================================================================================
# 4. ORGANISATION : blocs (questions, sections, cadres), champs, libelles, variables, codes
# =====================================================================================================================
def _libelles(ch, mots, autres_champs):
    """Les textes imprimes autour d'un champ : a gauche (meme ligne), dessous (legende), dessus (en-tete de colonne)."""
    x0, y0, x1, y1 = ch["bb"]
    cy = (y0 + y1) / 2
    gauche = [w for w in mots if w["x1"] <= x0 + 1 and x0 - w["x1"] < 160 and w["top"] - 2 <= cy <= w["bottom"] + 4
              and not any(c["bb"][0] >= w["x1"] - 1 and c["bb"][2] <= x0 + 1 and c["bb"][1] <= cy <= c["bb"][3]
                          for c in autres_champs if c is not ch)]
    # on garde le groupe de mots le plus proche du champ (ecart < 8 pt entre mots)
    gauche = sorted(gauche, key=lambda w: -w["x1"])
    run = []
    for w in gauche:
        if not run and x0 - w["x1"] > 40:
            break
        if run and run[-1]["x0"] - w["x1"] > 8:
            break
        run.append(w)
    run = list(reversed(run))
    dessous = [w for w in mots if y1 - 1 <= w["top"] <= y1 + 9 and w["x0"] < x1 + 2 and w["x1"] > x0 - 2]
    dessous = [w for w in dessous if abs(w["top"] - min(d["top"] for d in dessous)) < 2.5] if dessous else []
    dessus_c = [w for w in mots if w["bottom"] <= y0 + 1 and y0 - w["bottom"] < 45 and w["x0"] < x1 and w["x1"] > x0]
    dessus = []
    if dessus_c:
        bas = max(w["bottom"] for w in dessus_c)
        if not any(c["bb"][3] <= y0 + 1 and c["bb"][3] > bas and c["bb"][0] < x1 and c["bb"][2] > x0 for c in autres_champs):
            dessus = [w for w in dessus_c if abs(w["bottom"] - bas) < 2.5]
    return texte_de(run), texte_de(sorted(dessous, key=lambda w: w["x0"])), texte_de(sorted(dessus, key=lambda w: w["x0"]))


def organiser(o, largeur_unite):
    """A partir des objets d'une unite : la liste des BLOCS et des CHAMPS, chacun avec ses textes."""
    mots, var = o["mots"], o["variables"]
    champs = ([{**c, "type": "case"} for c in o["cases"]] + [{**c, "type": "texte"} for c in o["textes"]]
              + [{**c, "type": "ligne"} for c in o["lignes"]])
    # cellules a chiffres voisines sur une meme rangee -> un seul champ « chiffres » de n cases
    groupes = []
    for c in sorted(o["cellules"], key=lambda c: (round(c["bb"][1]), c["bb"][0])):
        g = groupes[-1] if groupes else None
        if g and abs(g[-1]["bb"][1] - c["bb"][1]) < 1.5 and 0 <= c["bb"][0] - g[-1]["bb"][2] < 0.45 * (c["bb"][2] - c["bb"][0]):
            g.append(c)
        else:
            groupes.append([c])
    champs += [{"bb": union(x["bb"] for x in g), "type": "chiffres", "cases": len(g),
                "cellules": [x["bb"] for x in g]} for g in groupes]
    # sous-titres : lignes en gras d au moins 7,5 pt, qui ne sont pas des legendes collees sous un champ
    legende = lambda w: any(0 <= w["top"] - c["bb"][3] < 9 and w["x0"] < c["bb"][2] and w["x1"] > c["bb"][0]
                            for c in champs if c["type"] != "case")
    gras_lignes = [l for l in lignes_de([w for w in mots if not legende(w)])
                   if sum(est_gras(w) and not est_italique(w) and w["size"] >= 7.5 for w in l) >= 0.5 * len(l)]
    for ch in champs:
        ch["libelle_gauche"], ch["libelle_dessous"], ch["libelle_dessus"] = _libelles(ch, mots, champs)
        # rubrique : la ligne en gras la plus proche au-dessus (ex. « Localisation du logement »)
        au_dessus = [l for l in gras_lignes if l[-1]["bottom"] <= ch["bb"][1] + 1 and ch["bb"][1] - l[-1]["bottom"] < 120
                     and l[0]["x0"] < ch["bb"][2] and l[-1]["x1"] > ch["bb"][0] - 200 and len(texte_de(l)) > 3]
        ch["rubrique"] = texte_de(max(au_dessus, key=lambda l: l[-1]["bottom"])) if au_dessus else ""
        vs = [w for w in var if dedans(centre(w), ch["bb"], 3)]
        ch["variables"] = [t.strip("()") for w in vs for t in w["text"].split()]
        ch["_vars"] = vs
        if ch["type"] == "case":
            x0, y0, x1, y1 = ch["bb"]
            cy = (y0 + y1) / 2
            d = [w for w in mots if re.fullmatch(r"\d{1,2}", w["text"]) and 0 <= w["x0"] - x1 < 12
                 and abs((w["top"] + w["bottom"]) / 2 - cy) < 6]
            ch["code"] = d[0]["text"] if d else None
    # suite d'un champ : une ligne / un champ sans libelle juste sous un champ qui en a un (adresse sur 2 lignes)
    for ch in sorted(champs, key=lambda c: c["bb"][1]):
        if ch["type"] in ("texte", "ligne") and not ch["libelle_gauche"]:
            au_dessus = [c for c in champs if c is not ch and c["type"] in ("texte", "ligne") and c["libelle_gauche"]
                         and -4 <= ch["bb"][1] - c["bb"][3] < 10 and c["bb"][0] < ch["bb"][2] and ch["bb"][0] < c["bb"][2]]
            if au_dessus:
                ch["libelle_gauche"] = au_dessus[0]["libelle_gauche"]
                ch["suite"] = True
    # ---- bandeaux : barres foncees larges et basses (titres de sections, ex. « A  Habitants permanents... »)
    cadres = o["cadres"]
    bandeaux = [c for c in cadres if c["couleur"] == "fonce" and c["bb"][3] - c["bb"][1] < 30
                and c["bb"][2] - c["bb"][0] > 0.5 * largeur_unite]

    def fin_section(bar):
        """Une section va de son bandeau jusqu'au bandeau suivant (meme colonne), sinon jusqu'au dernier element."""
        suiv = [b_["bb"][1] for b_ in bandeaux if b_["bb"][1] > bar[3] + 2 and b_["bb"][0] < bar[2] and b_["bb"][2] > bar[0]]
        # le cadre qui entoure le bandeau (s'il existe) borne la section : legendes du bas comprises
        autour = [c["bb"][3] for c in cadres if contient(c["bb"], bar, 1) and c["bb"] != bar and c["bb"][3] > bar[3] + 10]
        if suiv:
            return min([min(suiv) - 1.5] + autour)
        if autour:
            return min(autour)
        bas = [c["bb"][3] for c in champs + cadres if c["bb"][1] >= bar[3] - 1 and c["bb"][0] < bar[2] and c["bb"][2] > bar[0]]
        return max(bas) + 10 if bas else bar[3]

    blocs = []
    bars_pris = []
    # ---- tableaux : l'en-tete de colonne descend, le libelle de rangee se propage vers la droite
    for ch in sorted(champs, key=lambda c: c["bb"][1]):
        ch["colonne"] = ch["libelle_dessus"]
        if not ch["colonne"]:
            dessus = [c for c in champs if c is not ch and c["bb"][3] <= ch["bb"][1] + 1 and ch["bb"][1] - c["bb"][3] < 25
                      and abs((c["bb"][0] + c["bb"][2]) / 2 - (ch["bb"][0] + ch["bb"][2]) / 2) < 0.5 * (ch["bb"][2] - ch["bb"][0]) + 3
                      and c.get("colonne")]
            if dessus:
                ch["colonne"] = max(dessus, key=lambda c: c["bb"][3])["colonne"]
    for ch in sorted(champs, key=lambda c: c["bb"][0]):
        ch["rangee"] = ch["libelle_gauche"]
        if not ch["rangee"]:
            gauche = [c for c in champs if c is not ch and c["bb"][2] <= ch["bb"][0] + 1 and ch["bb"][0] - c["bb"][2] < 60
                      and abs((c["bb"][1] + c["bb"][3]) / 2 - (ch["bb"][1] + ch["bb"][3]) / 2) < 6 and c.get("rangee")]
            if gauche:
                ch["rangee"] = max(gauche, key=lambda c: c["bb"][2])["rangee"]
    # ---- blocs : chaque marqueur de question -> le plus petit cadre qui le porte (coin haut gauche)
    for m in [m for m in o["marqueurs"] if not m["renvoi"]]:
        mx0, my0, mx1, my1 = m["bb"]
        s = m["cote"]
        cand = [c for c in cadres if c["bb"][0] - 1.5 * s <= mx0 <= c["bb"][0] + 2 * s
                and c["bb"][1] - s <= my0 <= c["bb"][1] + 2 * s and c["bb"][2] > mx1 and c["bb"][3] > my1]
        cand.sort(key=lambda c: surface(c["bb"]))
        if not cand:
            continue
        cadre = cand[0]["bb"]
        if any(cand[0] is b_ for b_ in bandeaux):          # bandeau de section
            cadre = (cadre[0], cadre[1], cadre[2], fin_section(cadre))
            bars_pris.append(cand[0])
            sorte = "section"
        else:
            sorte = "question"
        blocs.append({"bb": union([cadre, m["bb"]]), "cadre": cadre, "marqueur": m, "numero": m["texte"], "sorte": sorte})
    # ---- bandeaux sans marqueur : sections nommees par leur texte (ou suite d'une section coupee par la pliure)
    for bar in bandeaux:
        if any(bar is b_ for b_ in bars_pris):
            continue
        cadre = (bar["bb"][0], bar["bb"][1], bar["bb"][2], fin_section(bar["bb"]))
        if cadre[3] - bar["bb"][3] < 10:
            continue                                       # simple barre de titre, sans rien dessous
        blocs.append({"bb": cadre, "cadre": cadre, "marqueur": None, "numero": None, "sorte": "section",
                      "_titre_bar": texte_de([w for l in lignes_de([w for w in mots if dedans(centre(w), bar["bb"])]) for w in l])})
    # ---- cadres sans numero qui contiennent des champs (identite, cadre de l'agent, localisation du logement...)
    pris = [b["cadre"] for b in blocs]
    for c in sorted(cadres, key=lambda c: surface(c["bb"])):
        if any(contient(c["bb"], p_) or contient(p_, c["bb"]) for p_ in pris):
            continue
        dedans_ch = [ch for ch in champs if contient(c["bb"], ch["bb"], 1)]
        if not dedans_ch or surface(c["bb"]) > 0.45 * largeur_unite * 1000:
            continue
        # titre : la ligne de texte juste au-dessus du cadre (ex. « Cadre a remplir par l'agent recenseur »)
        dessus = [l for l in lignes_de([w for w in mots if w["bottom"] <= c["bb"][1] + 1 and c["bb"][1] - w["bottom"] < 22
                                        and w["x0"] < c["bb"][2] and w["x1"] > c["bb"][0]])
                  if sum(est_gras(w) for w in l) >= 0.5 * len(l)]          # un titre en gras (pas « Exemple : »)
        blocs.append({"bb": c["bb"], "cadre": c["bb"], "marqueur": None, "numero": None, "sorte": "cadre",
                      "_titre_bar": texte_de(dessus[-1]) if dessus else ""})
        pris.append(c["bb"])
    # ---- rattacher chaque champ a son bloc (le plus petit qui le contient), textes du bloc
    for ch in champs:
        cont = [b for b in blocs if contient(b["bb"], ch["bb"], 1.5)]
        ch["_bloc"] = min(cont, key=lambda b: surface(b["bb"])) if cont else None
    # une « case » isolee, hors de tout bloc et sans code, est un motif decoratif (logo) : ignoree
    champs = [ch for ch in champs if ch["_bloc"] or not (ch["type"] == "case" and not ch.get("code"))]
    for b in blocs:
        b_mots = [w for w in mots if dedans(centre(w), b["bb"], 0.5)
                  and not (b["marqueur"] and dedans(centre(w), b["marqueur"]["bb"]))
                  and not any(dedans(centre(w), x["bb"]) for x in blocs if x is not b and contient(b["bb"], x["bb"]))]
        L = lignes_de(b_mots)
        titre, consignes = [], []
        for l in L:
            # debut gras de la ligne (« Quel est votre sexe ? » suivi de « Masculin 1 Feminin 2 » non gras)
            debut = []
            for w in l:
                if est_gras(w) and not est_italique(w):
                    debut.append(w)
                else:
                    break
            gras_ = len(debut) >= max(1, 0.5 * len(l)) or (len(debut) >= 2 and debut[-1]["text"].endswith(("?", ":")))
            if gras_ and not consignes and (not titre or l[0]["top"] - titre[-1][-1]["bottom"] < 8):
                titre.append(debut)
                if len(debut) < len(l):
                    break
                continue
            if titre and sum(est_italique(w) for w in l) >= 0.6 * len(l):
                consignes.append(l)
                continue
            if titre:
                break
        t = texte_de([w for l in titre for w in l])
        coupe = max(t.rfind("?"), t.rfind(" :"))
        if coupe > 0.4 * len(t):                      # « Quel est votre sexe ? Masculin Feminin » -> la question
            t = t[:coupe + (1 if t[coupe] == "?" else 2)]
        b["intitule"] = t if (t or not b.get("_titre_bar")) else b["_titre_bar"]
        if b["sorte"] == "section" and not b["numero"] and b.get("_titre_bar"):
            b["intitule"] = b["_titre_bar"]
        b["consignes"] = texte_de([w for l in consignes for w in l])
        b["zone_intitule"] = union(bb(w) for l in titre for w in l) if titre else None
        b["texte_complet"] = "\n".join(texte_de(l) for l in L)
        b["variables"] = []
        for w in sorted([w for w in var if dedans(centre(w), b["bb"], 3)], key=lambda w: (round(w["top"]), w["x0"])):
            b["variables"] += [t_.strip("()") for t_ in w["text"].split()]
        # case a cocher sans variable propre : la variable de la question (SEXE, MATR...) si elle est la seule « libre »
        libres = [t_.strip("()") for w in var if dedans(centre(w), b["bb"], 3)
                  and not any(w in ch["_vars"] for ch in champs) for t_ in w["text"].split()]
        for ch in champs:
            if ch["_bloc"] is b and ch["type"] == "case" and not ch["variables"] and len(libres) == 1:
                ch["variables"] = libres
    return blocs, champs


# =====================================================================================================================
# 5. CONSTRUCTION DE LA CONFIGURATION (tout le formulaire)
# =====================================================================================================================
def construire(pdf_source, formulaire, millesime, dpi=200, ordre="auto", source_nom=None):
    """Construit la configuration complete. Renvoie (config, images de reference des unites)."""
    pdf, octets = ouvrir_pdf(pdf_source)
    rendus = rendre_pages(octets, dpi)
    k = dpi / 72
    unites = unites_du_pdf(pdf, ordre)
    larg_px = max(round(u["largeur_pt"] * k) for u in unites)
    haut_px = [round(u["hauteur_pt"] * k) for u in unites]
    W, H = larg_px, sum(haut_px)
    debut = np.cumsum([0] + haut_px)
    blocs_cfg, champs_cfg, refs, reperes = [], [], [], []
    vus = {}

    def nouvel_id(base):
        vus[base] = vus.get(base, 0) + 1
        return base if vus[base] == 1 else f"{base}_{vus[base]}"

    def z(r, ui, marge=0.0):
        """Rectangle en points (repere de la page PDF) -> [x, y, w, h] en fractions de l'image EMPILEE."""
        u = unites[ui]
        x0, y0, x1, y1 = (r[0] - u["x0"] - marge) * k, (r[1] - marge) * k, (r[2] - u["x0"] + marge) * k, (r[3] + marge) * k
        x0, x1 = max(0, x0), min(W, x1)
        return [round(x0 / W, 5), round((y0 + debut[ui]) / H, 5), round((x1 - x0) / W, 5), round((y1 - y0) / H, 5)]

    parts = []
    for ui, u in enumerate(unites):
        p = pdf.pages[u["page_pdf"]]
        img = rendus[u["page_pdf"]]
        o = extraire_page(p, img, k, u["x0"], u["x1"])
        blocs, champs = organiser(o, u["largeur_pt"])
        parts.append((o, blocs, champs))
    # sections coupees par la pliure (tableau sur deux pages) : la moitie sans lettre reprend la lettre de l'autre
    lettre_par_y = {(unites[ui]["page_pdf"], round(b["cadre"][1])): b["numero"]
                    for ui, (_, bl, _) in enumerate(parts) for b in bl if b["sorte"] == "section" and b["numero"]}
    for ui, u in enumerate(unites):
        p = pdf.pages[u["page_pdf"]]
        img = rendus[u["page_pdf"]]
        o, blocs, champs = parts[ui]
        # image de reference de l'unite : rendu, noms de variables effaces (ils ne sont pas sur les vrais formulaires)
        a = img[:, int(round(u["x0"] * k)):int(round(u["x1"] * k))].copy()
        for w in o["variables"]:
            x0, y0 = int((w["x0"] - u["x0"]) * k) - 2, int(w["top"] * k) - 2
            x1, y1 = int((w["x1"] - u["x0"]) * k) + 2, int(w["bottom"] * k) + 2
            bord = np.concatenate([a[max(0, y0 - 3):max(0, y0), max(0, x0):x1].reshape(-1, 3),
                                   a[y1:y1 + 3, max(0, x0):x1].reshape(-1, 3)])
            a[max(0, y0):y1, max(0, x0):x1] = np.median(bord, axis=0) if len(bord) else 255
        if a.shape[1] < W:
            a = np.pad(a, ((0, 0), (0, W - a.shape[1]), (0, 0)), constant_values=255)
        refs.append(a[:haut_px[ui], :W])
        # reperes : traits epais courts aux coins (information ; le recalage n'en a pas besoin)
        rp = []
        for l in p.lines:
            if l.get("linewidth", 0) >= 1.5 and max(l["x1"] - l["x0"], l["bottom"] - l["top"]) < 16 \
                    and u["x0"] - 1 <= l["x0"] <= u["x1"] + 1:
                rp.append([round((l["x0"] - u["x0"]) / u["largeur_pt"], 4), round(l["top"] / u["hauteur_pt"], 4)])
        reperes.append(rp)
        # ---- blocs (ordre de lecture : colonne puis hauteur)
        col = lambda r: 0 if (r[0] - u["x0"]) < 0.45 * u["largeur_pt"] else 1
        for b in sorted(blocs, key=lambda b: (b["sorte"] != "cadre", col(b["bb"]), b["bb"][1])):
            if b["numero"]:
                base = ("q" if b["numero"].isdigit() else "section_") + b["numero"]
            elif b["sorte"] == "section":
                l_ = next((v for (pp, y), v in lettre_par_y.items() if pp == u["page_pdf"] and abs(y - b["cadre"][1]) < 3), None)
                base = f"section_{l_}_suite" if l_ else "section_" + slug(b["intitule"] or f"u{ui + 1}")
                if l_:
                    b["suite_de"] = f"section_{l_}"
            else:
                ch1 = sorted([c for c in champs if c["_bloc"] is b], key=lambda c: (c["bb"][1], c["bb"][0]))
                nom = b["intitule"] or b.get("_titre_bar") or next(
                    (c["libelle_gauche"] for c in ch1 if c["libelle_gauche"]), "") or next(
                    (c["libelle_dessus"] or c["libelle_dessous"] for c in ch1 if c["libelle_dessus"] or c["libelle_dessous"]), "")
                base = "cadre_" + slug(nom or f"u{ui + 1}")
            b["id"] = nouvel_id(base)
            blocs_cfg.append({
                "id": b["id"], "sorte": b["sorte"], "question": b["numero"], "unite": ui + 1,
                "numero_page_imprime": u["numero_imprime"],
                "intitule": b["intitule"], "consignes": b["consignes"], "texte_complet": b["texte_complet"],
                "zone": z(b["bb"], ui), "zone_intitule": z(b["zone_intitule"], ui) if b["zone_intitule"] else None,
                "marge": 0.004, "variables_formulaire": b["variables"], "parent": None, "champs": [],
                "theme": None, "intitule_court": None, "statut": "a_verifier",
                **({"suite_de": b["suite_de"]} if b.get("suite_de") else {})})
        for b in blocs:                                   # imbrication (un cadre dans une section...)
            parents = [x for x in blocs if x is not b and contient(x["bb"], b["bb"], 1)]
            if parents:
                pb = min(parents, key=lambda x: surface(x["bb"]))
                next(c for c in blocs_cfg if c["id"] == b["id"])["parent"] = pb["id"]
        # ---- champs (ordre de lecture dans leur bloc)
        compte = {}
        for ch in sorted(champs, key=lambda c: (c["_bloc"]["id"] if c["_bloc"] else "~", round(c["bb"][1] / 4), c["bb"][0])):
            bid = ch["_bloc"]["id"] if ch["_bloc"] else f"hors_bloc_u{ui + 1}"
            cle = (bid, ch["type"])
            compte[cle] = compte.get(cle, 0) + 1
            cid = f"{bid}_{ch['type']}{compte[cle]}"
            e = {"id": cid, "bloc": bid, "type": ch["type"], "unite": ui + 1, "zone": z(ch["bb"], ui, 1.0),
                 "libelle_gauche": ch["libelle_gauche"], "libelle_dessous": ch["libelle_dessous"],
                 "libelle_dessus": ch["libelle_dessus"], "rubrique": ch["rubrique"],
                 "colonne": ch["colonne"], "rangee": ch["rangee"],
                 "variables_formulaire": ch["variables"]}
            if ch.get("suite"):
                e["suite"] = True
            if ch["type"] == "case":
                e["code"] = ch.get("code")
            if ch["type"] == "chiffres":
                e["cases"] = ch["cases"]
                e["cellules"] = [z(c_, ui) for c_ in ch["cellules"]]
            champs_cfg.append(e)
            if ch["_bloc"]:
                next(c for c in blocs_cfg if c["id"] == bid)["champs"].append(cid)
    for b in blocs_cfg:
        b["reponse"] = nature_reponse([c for c in champs_cfg if c["bloc"] == b["id"]])
    nums = [int(b["question"]) for b in blocs_cfg if b["question"] and str(b["question"]).isdigit()]
    manquants = sorted(set(range(1, max(nums) + 1)) - set(nums)) if nums else []
    cfg = {"schema": VERSION_SCHEMA, "formulaire": formulaire, "millesime": millesime,
           "source": source_nom or (Path(pdf_source).name if not isinstance(pdf_source, (bytes, bytearray)) else "pdf"),
           "reference": {"largeur": int(W), "hauteur": int(H), "dpi": dpi, "source": "pdf_vierge"},
           "unites": [{"unite": u["unite"] + 1, "page_pdf": u["page_pdf"] + 1, "numero_imprime": u["numero_imprime"],
                       "fichier": None} for u in unites],
           "pages": [[round(debut[i] / H, 6), round(debut[i + 1] / H, 6)] for i in range(len(unites))],
           "images_reference": [], "reperes": reperes, "recalage": "orb_affine", "marge_defaut": 0.004,
           "controle": {"questions_trouvees": sorted(set(nums)), "questions_manquantes": manquants,
                        "n_blocs": len(blocs_cfg), "n_champs": len(champs_cfg),
                        "champs_par_type": {t: sum(c["type"] == t for c in champs_cfg)
                                            for t in ("case", "chiffres", "texte", "ligne")}},
           "blocs": blocs_cfg, "champs": champs_cfg}
    return cfg, refs


def nature_reponse(chs):
    t = {c["type"] for c in chs}
    if not t:
        return "aucune (information)"
    if t == {"case"}:
        return "cases a cocher"
    if "case" in t:
        return "ecriture + cases"
    return "ecriture" if t & {"texte", "ligne"} else "chiffres"


THEMES = ["identite", "adresse", "commune", "date", "naissance", "nationalite", "etudes", "situation_professionnelle",
          "profession", "etablissement", "activite", "transport", "logement", "habitants", "agent_recenseur",
          "sante", "couple", "information", "autre"]
MOTS_THEMES = [("adresse", ["ADRESSE", "LOCALISATION", "NOM DE LA VOIE", "NUMERO", "RESIDENCE HAMEAU"]),
               ("agent_recenseur", ["AGENT RECENSEUR"]), ("naissance", ["NAISSANCE", "NE LE", "NE E LE"]),
               ("nationalite", ["NATIONALITE"]), ("etudes", ["ENSEIGNEMENT", "DIPLOME", "ETUDES"]),
               ("profession", ["PROFESSION"]), ("etablissement", ["ETABLISSEMENT"]), ("transport", ["TRANSPORT"]),
               ("situation_professionnelle", ["SITUATION PRINCIPALE", "TRAVAILL", "EMPLOI", "CONTRAT", "SALARI", "TELETRAVAIL"]),
               ("habitants", ["HABITANTS", "PERSONNES", "ENFANTS"]), ("identite", ["NOM", "PRENOM", "SEXE"]),
               ("logement", ["LOGEMENT", "PIECES", "SURFACE", "BATIMENT", "CHAUFFAGE"]), ("sante", ["SANTE", "HANDICAP", "LIMITE"]),
               ("couple", ["COUPLE", "MARIE", "PACSE"])]


def theme_par_regles(bloc):
    """Theme sans modele (repli) : mots-cles dans l'intitule puis dans tout le texte du bloc."""
    if (bloc.get("reponse") or "").startswith("aucune"):
        return "information"
    for txt in (bloc.get("intitule") or "", bloc.get("texte_complet") or ""):
        t = normaliser(txt).replace("S ADRESSE", "")
        for theme, mots in MOTS_THEMES:
            if any(m in t for m in mots):
                return theme
    return "information" if bloc.get("reponse", "").startswith("aucune") else "autre"


# =====================================================================================================================
# 6. ENREGISTREMENT ET IMAGE DE CONTROLE
# =====================================================================================================================
def enregistrer(cfg, refs, dossier):
    dossier = Path(dossier)
    dossier.mkdir(parents=True, exist_ok=True)
    nom = f"{cfg['formulaire']}_{cfg['millesime']}"
    cfg["images_reference"] = []
    for i, a in enumerate(refs):
        f = f"{nom}_u{i + 1}.png"
        Image.fromarray(a).save(dossier / f, optimize=True)
        cfg["images_reference"].append(f)
        cfg["unites"][i]["fichier"] = f
    (dossier / f"{nom}.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=1))
    return dossier / f"{nom}.json"


def _police(t):
    for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "DejaVuSans-Bold.ttf", "arialbd.ttf"]:
        try:
            return ImageFont.truetype(f, t)
        except OSError:
            pass
    return ImageFont.load_default()


def image_controle(cfg, refs, montrer_champs=True):
    """Toutes les unites empilees ; chaque bloc encadre (rouge = question, bleu = section, violet = cadre) avec son id."""
    W, H = cfg["reference"]["largeur"], cfg["reference"]["hauteur"]
    can = Image.fromarray(np.vstack(refs))
    d = ImageDraw.Draw(can)
    F, f2 = _police(max(18, W // 50)), _police(max(11, W // 110))
    if montrer_champs:
        for c in cfg["champs"]:
            x, y, w, h = c["zone"]
            col = {"case": (0, 150, 0), "chiffres": (0, 120, 220), "texte": (0, 90, 220), "ligne": (230, 120, 0)}[c["type"]]
            d.rectangle([x * W, y * H, (x + w) * W, (y + h) * H], outline=col, width=2)
    for b in cfg["blocs"]:
        x, y, w, h = b["zone"]
        col = {"question": (220, 0, 0), "section": (0, 60, 200), "cadre": (150, 0, 170)}[b["sorte"]]
        d.rectangle([x * W, y * H, (x + w) * W, (y + h) * H], outline=col, width=4)
        lab = f"Q{b['question']}" if b["sorte"] == "question" else b["id"]
        if b["sorte"] == "cadre":
            d.text((x * W + 6, (y + h) * H - 4), lab, fill=col, font=f2, anchor="ld")
            continue
        tw = d.textlength(lab, font=F)
        d.rectangle([(x + w) * W - tw - 14, y * H + 3, (x + w) * W - 3, y * H + F.size + 10], fill=col)
        d.text(((x + w) * W - tw - 8, y * H + 5), lab, fill="white", font=F)
    return can


def catalogue(cfg):
    """Tableau de tous les elements (un par ligne) : pour l'explorer, le filtrer, l'exporter."""
    import pandas as pd
    lignes = []
    for b in cfg["blocs"]:
        lignes.append({"id": b["id"], "niveau": "bloc", "type": b["sorte"], "question": b["question"], "unite": b["unite"],
                       "intitule / libelle": b.get("intitule_court") or b["intitule"], "theme": b.get("theme"),
                       "reponse / code": b["reponse"], "variables": " ".join(b["variables_formulaire"]),
                       "parent": b.get("parent"), "statut": b.get("statut"), "zone": b["zone"]})
    for c in cfg["champs"]:
        lib = " / ".join(x for x in [c["libelle_gauche"], c["libelle_dessous"], c["libelle_dessus"]] if x)
        lignes.append({"id": c["id"], "niveau": "champ", "type": c["type"], "question": None, "unite": c["unite"],
                       "intitule / libelle": lib, "theme": None,
                       "reponse / code": c.get("code") or (f"{c['cases']} cases" if c.get("cases") else ""),
                       "variables": " ".join(c["variables_formulaire"]), "parent": c["bloc"], "statut": None,
                       "zone": c["zone"]})
    return pd.DataFrame(lignes)


# =====================================================================================================================
# 7. SELECTION : ce qu'un notebook d'etude veut decouper (sans toucher a la configuration)
# =====================================================================================================================
def selectionner(cfg, id=None, question=None, variables=None, libelle=None, rubrique=None, colonne=None, rangee=None, theme=None, titre=None, types=None,
                 avec_intitule=False, bloc_entier=False):
    """Choisit des elements et renvoie {"zone": union des zones, "elements": [ids], "blocs": [ids]}.
    Criteres (combinables) :
      id        : un ou plusieurs identifiants (bloc ou champ), ex. "q23" ou ["q23_texte1", "q23_texte2"]
      question  : numero de question (ex. 23) -> tout le bloc de la question
      variables : noms de variables imprimes (PDF « avec VAR »), ex. ["NUMVOI", "CPLADR"] -> les champs qui les portent
      libelle   : debut du libelle imprime a cote du champ (ou une liste), ex. "Adresse" -> « Adresse : » et sa suite
      rubrique  : sous-titre en gras au-dessus des champs, ex. "Localisation du logement"
      colonne, rangee : dans un tableau, l'en-tete de colonne (ex. "Nom") et le libelle de rangee (ex. 1 ou [1, 2])
      titre     : un mot de l'intitule du bloc, ex. "Localisation du logement"
      theme     : theme donne par le modele (ex. "adresse")
      types     : ne garder que ces types de champs (ex. ["texte", "ligne"])
    avec_intitule : ajoute l'intitule imprime de la question (et ce qui est entre lui et les champs : les consignes)
    bloc_entier   : prend le bloc entier des elements trouves."""
    blocs = {b["id"]: b for b in cfg["blocs"]}
    champs = {c["id"]: c for c in cfg["champs"]}
    pris = []
    if id is not None:
        for i in ([id] if isinstance(id, str) else id):
            if i not in blocs and i not in champs:
                raise KeyError(f"« {i} » n'existe pas dans le gabarit {cfg['formulaire']} {cfg['millesime']}")
            pris.append(i)
    if question is not None:
        qs = {str(q) for q in ([question] if isinstance(question, (str, int)) else question)}
        pris += [b["id"] for b in cfg["blocs"] if str(b["question"]) in qs]
    cand = list(champs.values())
    filtre = False
    if variables is not None:
        vs = {v.upper() for v in ([variables] if isinstance(variables, str) else variables)}
        cand = [c for c in cand if vs & {v.upper() for v in c["variables_formulaire"]}]
        filtre = True
    if libelle is not None:
        ns = [normaliser(x) for x in ([libelle] if isinstance(libelle, str) else libelle)]
        cand = [c for c in cand if any(normaliser(c.get(k) or "").startswith(n) for n in ns
                                       for k in ("libelle_gauche", "libelle_dessous", "libelle_dessus"))]
        filtre = True
    if colonne is not None:
        n = normaliser(colonne)
        cand = [c for c in cand if normaliser(c.get("colonne") or "").startswith(n)]
        filtre = True
    if rangee is not None:
        rs = {normaliser(str(x)) for x in ([rangee] if isinstance(rangee, (str, int)) else rangee)}
        cand = [c for c in cand if normaliser(c.get("rangee") or "") in rs]
        filtre = True
    if rubrique is not None:
        n = normaliser(rubrique)
        cand = [c for c in cand if n in normaliser(c.get("rubrique") or "")]
        filtre = True
    if titre is not None:
        n = normaliser(titre)
        ok = {b["id"] for b in cfg["blocs"] if n in normaliser(b["intitule"]) or n in normaliser(b.get("texte_complet", ""))}
        cand = [c for c in cand if c["bloc"] in ok] if filtre or theme else cand
        if not filtre and not theme:
            pris += sorted(ok, key=lambda i: surface_zone(blocs[i]["zone"]))[:1]
    if theme is not None:
        ok = {b["id"] for b in cfg["blocs"] if b.get("theme") == theme}
        cand = [c for c in cand if c["bloc"] in ok]
        filtre = True
    if types is not None:
        cand = [c for c in cand if c["type"] in types]
        filtre = True
    if filtre:
        pris += [c["id"] for c in cand]
    pris = list(dict.fromkeys(pris))
    if not pris:
        raise ValueError("Aucun element ne correspond a cette selection : verifier dans le catalogue (notebook 17).")
    blocs_concernes = list(dict.fromkeys(i if i in blocs else champs[i]["bloc"] for i in pris))
    zones = [blocs[i]["zone"] if i in blocs else champs[i]["zone"] for i in pris]
    if bloc_entier:
        zones += [blocs[b]["zone"] for b in blocs_concernes if b in blocs]
    if avec_intitule:
        for b in blocs_concernes:
            if b in blocs:
                zones.append(blocs[b]["zone_intitule"] or blocs[b]["zone"])
                if blocs[b]["zone_intitule"]:                 # du haut du bloc (numero compris) a l'intitule
                    x, y, w, h = blocs[b]["zone"]
                    zones.append([x, y, 0.001, blocs[b]["zone_intitule"][1] - y + blocs[b]["zone_intitule"][3]])
    xs0 = min(z_[0] for z_ in zones); ys0 = min(z_[1] for z_ in zones)
    xs1 = max(z_[0] + z_[2] for z_ in zones); ys1 = max(z_[1] + z_[3] for z_ in zones)
    if avec_intitule:                                          # largeur du bloc : l'intitule et les champs alignes
        for b in blocs_concernes:
            if b in blocs:
                xs0 = min(xs0, blocs[b]["zone"][0])
                xs1 = max(xs1, blocs[b]["zone"][0] + blocs[b]["zone"][2])
    que_des_champs = all(i in champs for i in pris) and not (avec_intitule or bloc_entier)
    return {"zone": [round(float(xs0), 5), round(float(ys0), 5), round(float(xs1 - xs0), 5), round(float(ys1 - ys0), 5)],
            "elements": pris, "blocs": blocs_concernes,
            "marge": 0.0015 if que_des_champs else cfg.get("marge_defaut", 0.004)}   # champs seuls : marge serree


def surface_zone(z_):
    return z_[2] * z_[3]


# =====================================================================================================================
# 8. RECALAGE D'UN SCAN SUR LA REFERENCE, ET DECOUPE
# =====================================================================================================================
class Recaleur:
    """Aligne chaque unite d'un scan sur l'unite vierge de reference (points ORB + RANSAC, transformation affine),
    puis decoupe des zones. Une unite qu'on ne peut pas recaler (autre formulaire...) n'est pas decoupee au hasard :
    elle est prise aux proportions et signalee (infos["recalee"] = False)."""

    LARGEUR_ORB = 1100
    MIN_POINTS = 25

    def __init__(self, cfg, dossier):
        import cv2
        self.cv2, self.cfg, self.dossier = cv2, cfg, Path(dossier)
        self._ref = {}

    def _orb(self):
        return self.cv2.ORB_create(nfeatures=6000, scaleFactor=1.2, nlevels=8, fastThreshold=10)

    def _petit(self, g):
        cv2 = self.cv2
        s = self.LARGEUR_ORB / g.shape[1]
        p = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(p), s

    def _gris(self, a):
        return a if a.ndim == 2 else self.cv2.cvtColor(a, self.cv2.COLOR_RGB2GRAY)

    def ref(self, i):
        if i not in self._ref:
            g = self._gris(np.array(Image.open(self.dossier / self.cfg["images_reference"][i]).convert("RGB")))
            p, s = self._petit(g)
            kp, de = self._orb().detectAndCompute(p, None)
            self._ref[i] = {"forme": g.shape, "s": s, "kp": kp, "de": de}
        return self._ref[i]

    def recaler_unite(self, a, i):
        cv2 = self.cv2
        R = self.ref(i)
        g = self._gris(a)
        p, s = self._petit(g)
        kp, de = self._orb().detectAndCompute(p, None)
        infos = {"unite": i + 1, "points": 0, "retenus": 0}
        if de is None or R["de"] is None or len(kp) < self.MIN_POINTS:
            return None, infos
        paires = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(de, R["de"], k=2)
        bons = [m for m in paires if len(m) == 2 and m[0].distance < 0.8 * m[1].distance]
        infos["points"] = len(bons)
        if len(bons) < self.MIN_POINTS:
            return None, infos
        src = np.float32([kp[m[0].queryIdx].pt for m in bons])
        dst = np.float32([R["kp"][m[0].trainIdx].pt for m in bons])
        A, garde = cv2.estimateAffine2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.0, maxIters=5000,
                                        confidence=0.999)
        if A is None:
            return None, infos
        infos["retenus"] = int(garde.sum())
        M = A.astype(np.float64)
        M[:, :2] *= s / R["s"]
        M[:, 2] /= R["s"]
        attendu = R["forme"][1] / g.shape[1]
        sx, sy = np.hypot(M[0, 0], M[1, 0]), np.hypot(M[0, 1], M[1, 1])
        angle = float(np.degrees(np.arctan2(M[1, 0], M[0, 0])))
        infos.update({"echelle_x": round(float(sx / attendu), 3), "echelle_y": round(float(sy / attendu), 3),
                      "angle": round(angle, 2)})
        if infos["retenus"] < self.MIN_POINTS or not (0.75 < sx / attendu < 1.33) or not (0.75 < sy / attendu < 1.33) \
                or abs(angle) > 10:
            return None, infos
        return M, infos

    def bornes_unites(self, img):
        """(haut, bas) de chaque unite dans le scan empile : coupure dans la bande la plus blanche pres de la
        position attendue (proportions des unites de reference)."""
        g = self._gris(img)
        h = g.shape[0]
        fr = [p_[1] for p_ in self.cfg["pages"][:-1]]
        if not fr:
            return [(0, h)]
        encre = np.convolve((g < 200).mean(axis=1), np.ones(15) / 15, mode="same")
        coupes = []
        for f in fr:
            c = f * h
            lo, hi = int(c - 0.04 * h), int(c + 0.04 * h)
            coupes.append(lo + int(np.argmin(encre[lo:hi] + 0.02 * np.abs(np.arange(lo, hi) - c) / (0.04 * h))))
        b = [0] + coupes + [h]
        return [(b[i], b[i + 1]) for i in range(len(b) - 1)]

    def toile(self, img):
        """Le scan entier ramene sur la reference (unites empilees, taille de la reference). Renvoie (toile, infos)."""
        cv2 = self.cv2
        img = np.asarray(img)
        img = img if img.ndim == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        W, H = self.cfg["reference"]["largeur"], self.cfg["reference"]["hauteur"]
        t = np.full((H, W, 3), 255, np.uint8)
        infos = []
        for i, (a, b) in enumerate(self.bornes_unites(img)):
            A, B = int(round(self.cfg["pages"][i][0] * H)), int(round(self.cfg["pages"][i][1] * H))
            m = int(0.04 * (b - a))
            a2, b2 = max(0, a - m), min(img.shape[0], b + m)
            M, inf = self.recaler_unite(img[a2:b2], i)
            if M is not None:
                t[A:B] = cv2.warpAffine(img[a2:b2], M, (W, B - A), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
                inf["recalee"] = True
                inf["_M"], inf["_debut"] = M, a2
            else:
                t[A:B] = cv2.resize(img[a:b], (W, B - A))
                inf["recalee"] = False
            infos.append(inf)
        return t, infos

    def decouper(self, toile, zone, marge=None):
        W, H = self.cfg["reference"]["largeur"], self.cfg["reference"]["hauteur"]
        marge = self.cfg.get("marge_defaut", 0.004) if marge is None else marge
        x, y, w, h = zone
        x0, y0 = max(0, x - marge), max(0, y - marge)
        x1, y1 = min(1, x + w + marge), min(1, y + h + marge)
        return toile[int(y0 * H):int(y1 * H), int(x0 * W):int(x1 * W)]


# =====================================================================================================================
# 9. TEST : faux scans (formulaire vierge + ecriture inventee, deformes) pour mesurer la precision du recalage
# =====================================================================================================================
def faux_scan(cfg, refs, echelle=0.75, angle=1.0, mode="couleur", graine=0):
    """Un scan SIMULE (aucune donnee reelle) : ecriture inventee dans les champs, chaque unite tournee, mise a
    l'echelle, decalee, bruitee, compressee en JPEG. Renvoie (image, transformations reference -> scan par unite)."""
    import cv2
    import io as _io
    import random
    rng = random.Random(graine)
    W, H = cfg["reference"]["largeur"], cfg["reference"]["hauteur"]
    pages = [Image.fromarray(r.copy()) for r in refs]
    textes = ["12 BIS RUE DES LILAS", "BAT C ESC 2", "DUPONT", "MARIE", "8 AV DE LA GARE", "LYON", "1984"]
    for c in cfg["champs"]:
        x, y, w, h = c["zone"]
        ui = c["unite"] - 1
        y0 = (y - cfg["pages"][ui][0]) * H
        d = ImageDraw.Draw(pages[ui])
        if c["type"] in ("texte", "ligne"):
            d.text((x * W + 8, y0 + 2), rng.choice(textes), fill=(20, 30, 120), font=_police(max(10, int(h * H * 0.65))))
        elif c["type"] == "case" and rng.random() < 0.33:
            d.line([(x * W, y0), ((x + w) * W, y0 + h * H)], fill=(20, 30, 120), width=3)
    scans, T = [], []
    for i, p in enumerate(pages):
        a = np.array(p)
        an = np.radians(angle + 0.3 * i)
        M = np.array([[echelle * np.cos(an), -echelle * np.sin(an), 25 + 7 * i],
                      [echelle * np.sin(an), echelle * np.cos(an), 18]])
        o = cv2.warpAffine(a, M, (int(a.shape[1] * echelle) + 50, int(a.shape[0] * echelle) + 50), borderValue=(250, 250, 250))
        if mode == "gris":
            o = cv2.cvtColor(cv2.cvtColor(o, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
        elif mode == "noir_blanc":
            g = cv2.cvtColor(o, cv2.COLOR_RGB2GRAY)
            o = cv2.cvtColor(np.where(g < 150, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
        o = np.clip(o + np.random.default_rng(graine + i).normal(0, 6, o.shape), 0, 255).astype(np.uint8)
        scans.append(o)
        T.append(M)
    larg = max(s.shape[1] for s in scans)
    img = np.vstack([np.pad(s, ((0, 0), (0, larg - s.shape[1]), (0, 0)), constant_values=250) for s in scans])
    b = _io.BytesIO()
    Image.fromarray(img).save(b, "JPEG", quality=60)
    return np.array(Image.open(b).convert("RGB")), T, [s.shape[0] for s in scans]


def erreur_recalage(cfg, infos, T, hauteurs):
    """Ecart maximal (pixels de la reference) entre la position vraie et la position recalee des coins des blocs."""
    W, H = cfg["reference"]["largeur"], cfg["reference"]["hauteur"]
    debut = np.cumsum([0] + list(hauteurs))
    errs = []
    for i, inf in enumerate(infos):
        if not inf["recalee"]:
            return float("inf")
        for b in cfg["blocs"]:
            if b["unite"] != i + 1:
                continue
            x, y, w, h = b["zone"]
            for px, py in [(x, y), (x + w, y + h)]:
                P = np.array([px * W, (py - cfg["pages"][i][0]) * H, 1.0])
                s_ = T[i] @ P
                Q = np.array([s_[0], s_[1] + debut[i] - inf["_debut"], 1.0])
                errs.append(float(np.hypot(*(inf["_M"] @ Q - P[:2]))))
    return max(errs) if errs else float("nan")


# =====================================================================================================================
# 10. POSITION D'UNE ZONE DANS LE SCAN D'ORIGINE (pour comparer avec une detection par modele), IoU
# =====================================================================================================================
def zone_dans_scan(cfg, infos_unite, zone):
    """Rectangle (x0, y0, x1, y1) en pixels du SCAN d'origine d'une zone du gabarit, grace au recalage de son unite.
    None si l'unite n'a pas ete recalee."""
    import cv2
    if not infos_unite.get("recalee"):
        return None
    W, H = cfg["reference"]["largeur"], cfg["reference"]["hauteur"]
    i = infos_unite["unite"] - 1
    x, y, w, h = zone
    y0_ref = (y - cfg["pages"][i][0]) * H
    pts = np.float32([[x * W, y0_ref], [(x + w) * W, y0_ref], [x * W, y0_ref + h * H], [(x + w) * W, y0_ref + h * H]])
    Minv = cv2.invertAffineTransform(infos_unite["_M"])
    p = pts @ Minv[:, :2].T + Minv[:, 2]
    p[:, 1] += infos_unite["_debut"]
    return (float(p[:, 0].min()), float(p[:, 1].min()), float(p[:, 0].max()), float(p[:, 1].max()))


def iou(a, b):
    """Recouvrement de deux rectangles (x0, y0, x1, y1) : 1 = identiques, 0 = disjoints."""
    if a is None or b is None:
        return float("nan")
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = surface(a) + surface(b) - inter
    return inter / u if u > 0 else 0.0


def blocs_a_decouper(cfg, quoi="toutes"):
    """Les blocs a decouper : "toutes" = chaque question / cadre / section qui attend une reponse ; ou une liste d'ids
    ou de numeros (ex. ["q23", "cadre_nom"] ou [23, 27])."""
    if quoi == "toutes":
        return [b for b in cfg["blocs"] if not b["reponse"].startswith("aucune")]
    voulus = {str(q) for q in quoi}
    return [b for b in cfg["blocs"] if b["id"] in voulus or str(b["question"]) in voulus]


def charger(fs, dossier, nom, local="/tmp/gabarits"):
    """Charge un gabarit depose (ex. dossier = "projet-mesure-qualite-rp/data/decoupes/gabarits", nom = "BI_2025") :
    copie locale de la configuration et des pages de reference, puis renvoie (configuration, Recaleur)."""
    local = Path(local)
    local.mkdir(parents=True, exist_ok=True)
    f_json = local / f"{nom}.json"
    fs.get(f"{dossier}/{nom}.json", str(f_json))
    cfg = json.loads(f_json.read_text())
    for f in cfg["images_reference"]:               # toujours recopiees : un gabarit refait remplace l'ancien
        fs.get(f"{dossier}/{f}", str(local / f))
    return cfg, Recaleur(cfg, local)
'''
f_code = TRAVAIL / "gabarits.py"
f_code.write_text(CODE_GABARITS)
sys.path.insert(0, str(TRAVAIL))
import gabarits as G
importlib.reload(G)
VERSION_CODE = hashlib.md5(CODE_GABARITS.encode()).hexdigest()[:10]
print(f"code commun charge (version {VERSION_CODE})")


In [ ]:
# 3. LES PDF VIERGES, dans l'espace du projet : liste, doublons, formulaire et millesime deduits du nom
def lister_pdfs(source):
    if fs.exists(source):
        if source.lower().endswith(".pdf"):
            return [source]
        return sorted(f for f in fs.find(source) if f.lower().endswith(".pdf"))
    print(f"« {source} » n'existe pas : recherche de tous les PDF « imprime » du projet (une minute environ)...")
    trouves = sorted(f for f in fs.find(BUCKET) if f.lower().endswith(".pdf") and "imprim" in Path(f).name.lower())
    if not trouves:
        raise FileNotFoundError(f"Aucun PDF vierge trouve dans {BUCKET}. Les deposer (ex. dans {BUCKET}/data/formulaires) "
                                f"et mettre ce chemin dans PDF_FORMULAIRES (cellule 1).")
    print("  trouves :", *trouves, sep="\n   ")
    return trouves


def lire_octets(chemin):
    with fs.open(chemin, "rb") as f:
        return f.read()


def deduire(nom):
    """« 2025-imprime-3-BI avec VAR.pdf » -> ("BI", 2025) ; « 2022-imprime-9-FLRM_NEW.pdf » -> ("FLRM", 2022)."""
    base = Path(nom).stem
    an = re.search(r"(?<!\d)(20\d\d)(?!\d)", base)
    m = re.search(r"imprim[eé][-_ ]*\d*[-_ ]+([A-Za-z]+)", base, re.I) or re.search(r"\b(BI|FLRM|FLAG|FLR|FL)\b", base.upper())
    return (FORMULAIRE or (m.group(1).upper() if m else base[:10]), MILLESIME or (int(an.group(1)) if an else None))


lignes_, vus_ = [], {}
for chemin in lister_pdfs(PDF_FORMULAIRES):
    octets = lire_octets(chemin)
    md5 = hashlib.md5(octets).hexdigest()
    form, mill = deduire(chemin)
    lignes_.append({"fichier": Path(chemin).name, "formulaire": form, "millesime": mill, "taille_ko": len(octets) // 1024,
                    "avec_VAR": "VAR" in Path(chemin).name.upper(), "doublon_de": vus_.get(md5), "chemin": chemin})
    vus_.setdefault(md5, Path(chemin).name)
DOCS = pd.DataFrame(lignes_)
A_TRAITER = (DOCS[DOCS.doublon_de.isna() & DOCS.millesime.notna()]
             .sort_values(["formulaire", "millesime", "avec_VAR"], ascending=[True, True, False])
             .drop_duplicates(["formulaire", "millesime"]))
with pd.option_context("display.width", 200, "display.max_colwidth", 70):
    display(DOCS.drop(columns="chemin"))
print(f"\n{len(A_TRAITER)} formulaire(s) : " + ", ".join(f"{r.formulaire} {r.millesime}" for r in A_TRAITER.itertuples())
      + (f" | {int(DOCS.doublon_de.notna().sum())} doublon(s) ignore(s)" if DOCS.doublon_de.notna().any() else ""))


In [ ]:
# 4. CONSTRUCTION AUTOMATIQUE : tous les elements de chaque formulaire, lus dans le PDF (positions exactes)
#    Controles immediats : numeros de questions qui se suivent (1 a N sans trou), nombre d'elements par type.
PDF_FORMULAIRES = f"{BUCKET}/documentation/Questionnaires/2025-imprime-3-BI - avec VAR.pdf"   # le BI 2025 seul
GABS = {}                                   # "BI_2025" -> {"cfg": configuration, "refs": images de reference}
resume_ = []
for r in A_TRAITER.itertuples():
    t0 = time.time()
    cfg, refs = G.construire(lire_octets(r.chemin), r.formulaire, int(r.millesime), dpi=DPI, ordre=ORDRE_SCAN,
                             source_nom=r.fichier)
    nom = f"{r.formulaire}_{int(r.millesime)}"
    GABS[nom] = {"cfg": cfg, "refs": refs}
    c_ = cfg["controle"]
    resume_.append({"gabarit": nom, "unites (pages scannees)": len(cfg["unites"]), "blocs": c_["n_blocs"],
                    "questions": len(c_["questions_trouvees"]), "questions manquantes": c_["questions_manquantes"] or "",
                    **{f"champs {k}": v for k, v in c_["champs_par_type"].items()},
                    "variables imprimees": len({v for b in cfg["blocs"] for v in b["variables_formulaire"]}),
                    "secondes": round(time.time() - t0, 1)})
RESUME = pd.DataFrame(resume_)
with pd.option_context("display.width", 250):
    display(RESUME)
for nom, g_ in GABS.items():
    if g_["cfg"]["controle"]["questions_manquantes"]:
        print(f"ATTENTION {nom} : questions non trouvees {g_['cfg']['controle']['questions_manquantes']} "
              f"(voir l'image de controle : un carre numerote dessine autrement ?)")
# apercu : les blocs du premier formulaire
nom0 = next(iter(GABS), None)
if nom0:
    with pd.option_context("display.width", 250, "display.max_colwidth", 70, "display.max_rows", 60):
        display(pd.DataFrame([{"id": b["id"], "sorte": b["sorte"], "page": b["unite"], "intitule": b["intitule"][:70],
                               "reponse": b["reponse"], "champs": len(b["champs"]),
                               "variables": " ".join(b["variables_formulaire"])[:40]} for b in GABS[nom0]["cfg"]["blocs"]]))


In [ ]:
# 5. LE SENS, PAR LE MODELE (gemma4, gratuit) : theme et intitule court de chaque bloc, a partir de son texte.
#    Sans modele (ou s'il ne repond pas), des regles par mots-cles prennent le relais. Reponses gardees en cache.
GABARITS_A_TRAITER = ["BI_2025"]    # les gabarits a traiter ici ; None = tous ceux construits en cellule 4

CACHE_MODELE = TRAVAIL / "cache_modele_gabarits.json"
CACHE_M = json.loads(CACHE_MODELE.read_text()) if CACHE_MODELE.exists() else {}


def demander_gemma(contenu, delai=120, max_tokens=600, modele=None):
    """Un appel a un modele interne (gemma4 par defaut) ; renvoie le JSON de la reponse (dict) ou None."""
    modele = modele or MODELE_GEMMA
    cle = hashlib.md5(json.dumps([modele, contenu], ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if cle in CACHE_M:
        return CACHE_M[cle]
    corps = {"model": modele, "messages": [{"role": "user", "content": contenu}], "temperature": 0,
             "max_tokens": max_tokens}
    for essai in range(3):
        try:
            r = requests.post(URL_SSP, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
                              json=corps, timeout=delai)
            if r.status_code == 429:
                time.sleep(5 * (essai + 1))
                continue
            r.raise_for_status()
            texte = r.json()["choices"][0]["message"]["content"] or ""
            m = re.search(r"\{.*\}", texte, re.S)
            rep = json.loads(m.group(0)) if m else None
            CACHE_M[cle] = rep
            return rep
        except Exception:
            time.sleep(2)
    return None


CONSIGNE_SENS = """Voici un bloc d'un formulaire du recensement de la population francais ({form} {mill}).
Identifiant : {id} ; sorte : {sorte} ; numero : {num}
Texte imprime du bloc :
{texte}
Champs a remplir : {champs}

Reponds uniquement en JSON :
{{"theme": "<un seul parmi : {themes}>",
  "intitule_court": "<4 a 8 mots qui resument ce qui est demande>",
  "description": "<une phrase : ce que la personne ecrit ou coche ici>"}}"""


def sens_du_bloc(cfg, b):
    chs = [c for c in cfg["champs"] if c["bloc"] == b["id"]]
    resume_ch = "; ".join(f"{c['type']}" + (f" « {c['libelle_gauche'] or c['libelle_dessous'] or c['libelle_dessus']} »"
                                             if (c['libelle_gauche'] or c['libelle_dessous'] or c['libelle_dessus']) else "")
                          for c in chs[:25]) or "aucun"
    rep = demander_gemma(CONSIGNE_SENS.format(form=cfg["formulaire"], mill=cfg["millesime"], id=b["id"], sorte=b["sorte"],
                                              num=b["question"] or "-", texte=b["texte_complet"][:1500],
                                              champs=resume_ch, themes=", ".join(G.THEMES)))
    if isinstance(rep, dict) and rep.get("theme") in G.THEMES:
        return {"theme": rep["theme"], "intitule_court": str(rep.get("intitule_court", ""))[:80],
                "description": str(rep.get("description", ""))[:200], "sens_par": MODELE_GEMMA}
    return {"theme": G.theme_par_regles(b), "intitule_court": (b["intitule"] or b["id"])[:80], "description": "",
            "sens_par": "regles"}


a_traiter_ = {nom: g_ for nom, g_ in GABS.items() if GABARITS_A_TRAITER is None or nom in GABARITS_A_TRAITER}
ignores_ = [nom for nom in GABS if nom not in a_traiter_]
if ignores_:
    print(f"ignores (hors GABARITS_A_TRAITER) : {', '.join(ignores_)}")
if not a_traiter_:
    raise RuntimeError(f"aucun gabarit {GABARITS_A_TRAITER} parmi ceux construits en cellule 4 : {list(GABS)}")

for nom, g_ in a_traiter_.items():
    cfg = g_["cfg"]
    if UTILISER_MODELE:
        with ThreadPoolExecutor(N_PARALLELE) as ex:
            sens_ = list(ex.map(lambda b: sens_du_bloc(cfg, b), cfg["blocs"]))
    else:
        sens_ = [{"theme": G.theme_par_regles(b), "intitule_court": (b["intitule"] or b["id"])[:80], "description": "",
                  "sens_par": "regles"} for b in cfg["blocs"]]
    for b, s_ in zip(cfg["blocs"], sens_):
        b.update(s_)
    CACHE_MODELE.write_text(json.dumps(CACHE_M, ensure_ascii=False))
    par = pd.Series([b["sens_par"] for b in cfg["blocs"]]).value_counts().to_dict()
    print(f"{nom} : sens donne par {par}")
    with pd.option_context("display.width", 250, "display.max_colwidth", 60, "display.max_rows", 80):
        display(pd.DataFrame([{"id": b["id"], "theme": b["theme"], "intitule court": b["intitule_court"]}
                              for b in cfg["blocs"]]))

In [ ]:
# 6. VERIFICATION AUTOMATIQUE de chaque bloc du gabarit
#    a) controles geometriques (sans modele) : deux blocs de meme sorte ne doivent pas se chevaucher ;
#    b) gemma4 REGARDE la decoupe du bloc sur le formulaire vierge : la question est-elle entiere, et seule ?
#    statut : « valide » (tout est bon), « a_verifier » (a regarder sur l'image de controle), avec la raison.
CONSIGNE_VUE = """Cette image est un morceau d'un formulaire vierge du recensement.
Elle devrait montrer {quoi}.
Reponds uniquement en JSON :
{{"numero_visible": "<le numero de question visible en haut a gauche, ou VIDE>",
  "bloc_entier": <true si le bloc est entier (titre et toutes les cases ou lignes de reponse), false s'il est coupe>,
  "autre_question_visible": <true si une AUTRE question (autre numero) apparait aussi>}}"""


def image_bloc(cfg, refs, zone, marge=0.004):
    H, W = cfg["reference"]["hauteur"], cfg["reference"]["largeur"]
    toile = np.vstack(refs)
    x, y, w, h = zone
    return toile[max(0, int((y - marge) * H)):int((y + h + marge) * H), max(0, int((x - marge) * W)):int((x + w + marge) * W)]


def en_base64(a, cote_max=1400):
    im = Image.fromarray(a)
    r = cote_max / max(im.size)
    if r < 1:
        im = im.resize((int(im.width * r), int(im.height * r)))
    b = io.BytesIO()
    im.save(b, format="PNG")
    return base64.b64encode(b.getvalue()).decode()


def verifier_bloc(cfg, refs, b):
    raisons = []
    meme = [x for x in cfg["blocs"] if x is not b and x["sorte"] == b["sorte"] and x["unite"] == b["unite"]]
    for x in meme:
        bx, xx = b["zone"], x["zone"]
        inter = max(0, min(bx[0] + bx[2], xx[0] + xx[2]) - max(bx[0], xx[0])) * max(0, min(bx[1] + bx[3], xx[1] + xx[3]) - max(bx[1], xx[1]))
        if inter > 0.1 * bx[2] * bx[3]:
            raisons.append(f"chevauche {x['id']}")
    vue = None
    if VERIFIER_PAR_IMAGE:
        quoi = (f"la question {b['question']} « {b['intitule'][:120]} »" if b["sorte"] == "question"
                else f"le bloc « {(b.get('intitule_court') or b['intitule'] or b['id'])[:120]} »")
        vue = demander_gemma([{"type": "text", "text": CONSIGNE_VUE.format(quoi=quoi)},
                              {"type": "image_url", "image_url": {"url": "data:image/png;base64,"
                                                                 + en_base64(image_bloc(cfg, refs, b["zone"]))}}])
        if not isinstance(vue, dict):
            raisons.append("modele sans reponse")
        else:
            if vue.get("bloc_entier") is False:
                raisons.append("le modele voit un bloc coupe")
            if vue.get("autre_question_visible") is True:
                raisons.append("le modele voit une autre question")
            nv = re.sub(r"\D", "", str(vue.get("numero_visible", "")))
            if b["sorte"] == "question" and nv and nv != str(b["question"]):
                raisons.append(f"le modele lit le numero {nv}")
    return {"statut": "valide" if not raisons else "a_verifier", "verification": "; ".join(raisons) or "ok",
            "verifie_par": (MODELE_GEMMA if VERIFIER_PAR_IMAGE else "controles geometriques")}


for nom, g_ in GABS.items():
    cfg, refs = g_["cfg"], g_["refs"]
    with ThreadPoolExecutor(N_PARALLELE) as ex:
        res_ = list(ex.map(lambda b: verifier_bloc(cfg, refs, b), cfg["blocs"]))
    for b, r_ in zip(cfg["blocs"], res_):
        b.update(r_)
    CACHE_MODELE.write_text(json.dumps(CACHE_M, ensure_ascii=False))
    av = [b for b in cfg["blocs"] if b["statut"] != "valide"]
    cfg["controle"]["blocs_valides"] = len(cfg["blocs"]) - len(av)
    print(f"{nom} : {len(cfg['blocs']) - len(av)}/{len(cfg['blocs'])} blocs valides"
          + (" | a verifier : " + "; ".join(f"{b['id']} ({b['verification']})" for b in av[:12]) if av else ""))


In [ ]:
# 7. LE FORMULAIRE A-T-IL EVOLUE ? Comparaison avec le millesime precedent du meme formulaire deja depose dans
#    data/decoupes/gabarits : questions ajoutees, supprimees, renumerotees (rapprochees par VARIABLE, sinon par intitule).
GAB_S3 = f"{DECOUPES}/gabarits"
MEME_FORMULAIRE = {"FLR": ["FLRM"], "FLRM": ["FLR"]}     # un formulaire qui change de nom d'un millesime a l'autre


def cle_question(b):
    return ("var:" + "+".join(sorted(b["variables_formulaire"]))) if b["variables_formulaire"] else "txt:" + G.normaliser(b["intitule"])[:60]


deposes = [f for f in (fs.find(GAB_S3) if fs.exists(GAB_S3) else []) if f.endswith(".json")]
EVOLUTIONS = {}
for nom, g_ in GABS.items():
    cfg = g_["cfg"]
    familles = {cfg["formulaire"], *MEME_FORMULAIRE.get(cfg["formulaire"], [])}
    anciens = sorted([f for f in deposes if Path(f).stem.rsplit("_", 1)[0] in familles
                      and Path(f).stem.rsplit("_", 1)[-1].isdigit() and int(Path(f).stem.rsplit("_", 1)[-1]) < cfg["millesime"]],
                     key=lambda f: int(Path(f).stem.rsplit("_", 1)[-1]))
    if not anciens:
        print(f"{nom} : pas de millesime precedent depose (premiere configuration de ce formulaire)")
        continue
    with fs.open(anciens[-1], "r") as f_:
        prec = json.load(f_)
    q_new = {cle_question(b): b for b in cfg["blocs"] if b["sorte"] == "question"}
    q_old = {cle_question(b): b for b in prec["blocs"] if b["sorte"] == "question"}
    ev = ([{"changement": "ajoutee", "question": b["question"], "intitule": b["intitule"][:80]}
           for k, b in q_new.items() if k not in q_old]
          + [{"changement": "supprimee", "question": b["question"], "intitule": b["intitule"][:80]}
             for k, b in q_old.items() if k not in q_new]
          + [{"changement": f"renumerotee ({q_old[k]['question']} -> {b['question']})", "question": b["question"],
              "intitule": b["intitule"][:80]} for k, b in q_new.items() if k in q_old and q_old[k]["question"] != b["question"]])
    EVOLUTIONS[nom] = pd.DataFrame(ev)
    print(f"{nom} compare a {Path(anciens[-1]).stem} : {len(ev)} changement(s)")
    if ev:
        with pd.option_context("display.width", 200, "display.max_colwidth", 80):
            display(EVOLUTIONS[nom])


In [ ]:
# 8. DEPOT DU GABARIT la ou vont les decoupes : data/decoupes/gabarits/
#    <FORM>_<ANNEE>.json (tous les elements), <FORM>_<ANNEE>_u1.png... (pages vierges de reference), _controle.jpg,
#    catalogue_<FORM>_<ANNEE>.xlsx (tous les elements, un par ligne) et gabarits.py (le code commun).
#    Aucune donnee : uniquement le formulaire VIERGE.
# --- seulement les gabarits demandes en cellule 1 (ex. BI_2025) : les cellules 8 a 14 ne voient plus que ceux-la,
#     meme si d'autres ont ete construits plus tot dans la session
GABARITS_A_TRAITER = globals().get("GABARITS_A_TRAITER", ["BI_2025"])   # reglage de la cellule 1
GABS = {n: g for n, g in GABS.items() if not GABARITS_A_TRAITER or n in GABARITS_A_TRAITER}
if not GABS:
    raise RuntimeError(f"aucun gabarit {GABARITS_A_TRAITER} construit : relancer les cellules 3 et 4")
DECOUPER = {f: s for f, s in DECOUPER.items() if any(g["cfg"]["formulaire"] == f for g in GABS.values())}
print(f"gabarits traites a partir d'ici : {', '.join(GABS)} | formulaires a decouper : {', '.join(DECOUPER) or 'aucun'}")

FICHIERS = {}
local_ = TRAVAIL / "gabarits"
for nom, g_ in GABS.items():
    cfg, refs = g_["cfg"], g_["refs"]
    cfg["cree_le"] = date.today().isoformat()
    cfg["version_code"] = VERSION_CODE
    cfg["construit_par"] = "notebook 17 (PDF vierge" + (f" + {MODELE_GEMMA}" if UTILISER_MODELE or VERIFIER_PAR_IMAGE else "") + ")"
    f_json = G.enregistrer(cfg, refs, local_)
    f_ctrl = local_ / f"{nom}_controle.jpg"
    ctrl_ = G.image_controle(cfg, refs)
    ctrl_.thumbnail((1200, 100000))
    ctrl_.convert("RGB").save(f_ctrl, quality=85, optimize=True)
    f_cat = local_ / f"catalogue_{nom}.xlsx"
    G.catalogue(cfg).to_excel(f_cat, index=False)
    for f_ in [f_json, f_ctrl, f_cat] + [local_ / x for x in cfg["images_reference"]]:
        fs.put(str(f_), f"{GAB_S3}/{f_.name}")
    FICHIERS[nom] = {"json": f_json, "controle": f_ctrl, "catalogue": f_cat}
    print(f"{nom} -> {GAB_S3}/{f_json.name} (+ {len(cfg['images_reference'])} pages de reference, controle, catalogue)")
fs.put(str(f_code), f"{GAB_S3}/gabarits.py")
print(f"code commun -> {GAB_S3}/gabarits.py")
nom0 = next(iter(GABS), None)
if nom0:
    im_ = Image.open(FICHIERS[nom0]["controle"])
    display(im_.resize((900, int(900 * im_.height / im_.width))))
    print("Rouge = question, bleu = section de tableau, violet = cadre sans numero ; petits cadres = champs "
          "(vert = case, bleu = ecriture / chiffres, orange = ligne d'ecriture).")

In [ ]:
# 9. LE RECALAGE MARCHE-T-IL ? Test sur de FAUX scans (formulaire vierge + ecriture inventee : aucune donnee) :
#    papier penche, scanne a 150 / 200 / 300 dpi, en couleur, en gris, en noir et blanc, bruite, compresse en JPEG.
#    On connait la deformation appliquee : on mesure l'ecart entre la position vraie et la position recalee des blocs.
ESSAIS_FAUX = [(0.75, 1.5, "couleur"), (1.0, -2.0, "gris"), (1.5, 0.8, "couleur"), (0.75, 0.6, "noir_blanc"),
               (0.6, -1.0, "gris")]
lignes_ = []
for nom, g_ in GABS.items():
    R_ = G.Recaleur(g_["cfg"], local_)
    for k_, (ech, ang, mode) in enumerate(ESSAIS_FAUX):
        img_, T_, hs_ = G.faux_scan(g_["cfg"], g_["refs"], ech, ang, mode, graine=k_)
        t0 = time.time()
        toile_, infos_ = R_.toile(img_)
        err = G.erreur_recalage(g_["cfg"], infos_, T_, hs_)
        lignes_.append({"gabarit": nom, "dpi du scan": int(DPI * ech), "angle": ang, "mode": mode,
                        "ecart max (mm)": round(err / DPI * 25.4, 2) if np.isfinite(err) else None,
                        "pages recalees": f"{sum(i['recalee'] for i in infos_)}/{len(infos_)}",
                        "secondes": round(time.time() - t0, 1)})
TEST_FAUX = pd.DataFrame(lignes_)
display(TEST_FAUX)
for nom in GABS:
    e_ = TEST_FAUX[TEST_FAUX.gabarit == nom]["ecart max (mm)"]
    GABS[nom]["cfg"]["controle"]["recalage_faux_scans_ecart_max_mm"] = float(e_.max()) if e_.notna().all() else None
    print(f"{nom} : ecart maximal {e_.max()} mm" + ("" if e_.notna().all() and e_.max() < 1 else "  <- A REGARDER"))


In [ ]:
# 10. LES MODELES SAVENT-ILS TROUVER UNE QUESTION SUR UN VRAI SCAN ? Comparaison avec le recalage sur le PDF.
#     Chaque modele de vision interne recoit la page et doit donner le cadre de la question (numero + intitule fournis).
#     On mesure le recouvrement (IoU : 1 = meme cadre, 0 = rien en commun) avec le cadre donne par le recalage,
#     la part de reponses exploitables et le temps. Le meilleur (IoU moyen >= 0,7) sert de SECOURS a la cellule 11.
#     Images envoyees : uniquement aux modeles INTERNES du SSP Cloud.
CONSIGNE_DETECTION = """Voici une page scannee d'un bulletin du recensement de la population, rempli a la main.
Trouve le cadre de la question {num} : « {intitule} ».
Le numero {num} est ecrit dans un petit carre fonce en haut a gauche du cadre.
Donne le rectangle qui contient TOUTE cette question (numero, texte imprime, cases et zones de reponse),
en fractions de la largeur et de la hauteur de l'image (nombres entre 0 et 1).
Reponds uniquement en JSON : {{"x0": ..., "y0": ..., "x1": ..., "y1": ...}}"""


def modeles_vision():
    if MODELES_DETECTION != "auto":
        return list(MODELES_DETECTION)
    try:
        r = requests.get(URL_SSP_MODELES, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}, timeout=30)
        ids = [m.get("id") for m in r.json().get("data", [])]
    except Exception as e:
        print(f"liste des modeles internes indisponible ({type(e).__name__}) : gemma4 seul")
        ids = []
    vision = [i for i in ids if i and re.search(r"gemma|vl|vision|pixtral|llava|llama-4|minicpm|internvl|mistral-small", i, re.I)
              and not re.search(r"embed|rerank|whisper|ocr|chandra", i, re.I)]
    return list(dict.fromkeys([MODELE_GEMMA] + vision))


def detecter(modele, page, b):
    """Cadre (x0, y0, x1, y1) en pixels de la page, donne par le modele ; (None, secondes) si inexploitable."""
    im = Image.fromarray(page)
    r_ = 1000 / max(im.size)
    im = im.resize((int(im.width * r_), int(im.height * r_)))
    b64 = en_base64(np.array(im), 1000)
    t0 = time.time()
    rep = demander_gemma([{"type": "text", "text": CONSIGNE_DETECTION.format(num=b["question"] or b["id"],
                                                                            intitule=b["intitule"][:150])},
                          {"type": "image_url", "image_url": {"url": "data:image/png;base64," + b64}}],
                         modele=modele, max_tokens=300)
    s_ = time.time() - t0
    try:
        x0, y0, x1, y1 = (float(rep[k]) for k in ("x0", "y0", "x1", "y1"))
    except Exception:
        return None, s_
    if max(x0, y0, x1, y1) > 1.5:                      # reponse en pixels de l'image envoyee
        x0, x1, y0, y1 = x0 / im.width, x1 / im.width, y0 / im.height, y1 / im.height
    if not (0 <= x0 < x1 <= 1.05 and 0 <= y0 < y1 <= 1.05):
        return None, s_
    H_, W_ = page.shape[:2]
    return (x0 * W_, y0 * H_, x1 * W_, y1 * H_), s_


def scans_du_formulaire(suffixe):
    return sorted(f for f in fs.find(SCANS) if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
                  and Path(f).stem.upper().endswith("_" + suffixe.upper()))


def lire_image(chemin):
    with fs.open(chemin, "rb") as f_:
        return np.array(Image.open(io.BytesIO(f_.read())).convert("RGB"))


def gabarit_pour(form):
    """Le gabarit du formulaire pour la campagne : millesime = annee de la campagne, sinon le plus recent avant."""
    an = 2000 + int(re.sub(r"\D", "", CAMPAGNE)[-2:])
    noms = sorted([n for n, g_ in GABS.items() if g_["cfg"]["formulaire"] == form], key=lambda n: GABS[n]["cfg"]["millesime"])
    avant = [n for n in noms if GABS[n]["cfg"]["millesime"] <= an]
    return (avant or noms or [None])[-1]


SCANS_PAR_FORM, DETECTION, MEILLEUR_MODELE = {}, {}, {}
for form, suffixe in DECOUPER.items():
    nom = gabarit_pour(form)
    if not nom:
        print(f"{form} : pas de gabarit (PDF vierge absent) - rien a decouper")
        continue
    SCANS_PAR_FORM[form] = scans_du_formulaire(suffixe)
    print(f"{form} : {len(SCANS_PAR_FORM[form])} scans « _{suffixe} » dans {SCANS} | gabarit {nom}")
    if not SCANS_PAR_FORM[form] or not N_SCANS_DETECTION:
        continue
    cfg = GABS[nom]["cfg"]
    R_ = G.Recaleur(cfg, local_)
    qs = [b for b in cfg["blocs"] if b["id"] in QUESTIONS_DETECTION] or G.blocs_a_decouper(cfg)[:6]
    modeles = modeles_vision()
    print(f"  modeles compares : {modeles}")
    taches = []
    for chemin in SCANS_PAR_FORM[form][:N_SCANS_DETECTION]:
        img = lire_image(chemin)
        toile_, infos_ = R_.toile(img)
        bornes = R_.bornes_unites(img)
        for b in qs:
            inf = infos_[b["unite"] - 1]
            vrai = G.zone_dans_scan(cfg, inf, b["zone"])
            if vrai is None:
                continue
            a_, b_ = bornes[b["unite"] - 1]
            for m in modeles:
                taches.append((m, chemin, b, img[a_:b_], a_, vrai))
    lignes_ = []

    def faire(t):
        m, chemin, b, page, a_, vrai = t
        boite, s_ = detecter(m, page, b)
        if boite:
            boite = (boite[0], boite[1] + a_, boite[2], boite[3] + a_)
        return {"modele": m, "scan": Path(chemin).name, "bloc": b["id"], "reponse_ok": boite is not None,
                "iou": G.iou(boite, vrai) if boite else 0.0, "secondes": round(s_, 1)}

    with ThreadPoolExecutor(N_PARALLELE) as ex:
        lignes_ = list(ex.map(faire, taches))
    CACHE_MODELE.write_text(json.dumps(CACHE_M, ensure_ascii=False))
    D_ = pd.DataFrame(lignes_)
    if len(D_):
        T_ = (D_.groupby("modele").agg(appels=("iou", "size"), reponses_exploitables=("reponse_ok", "mean"),
                                       iou_moyen=("iou", "mean"), iou_sup_0_5=("iou", lambda x: (x >= 0.5).mean()),
                                       iou_sup_0_7=("iou", lambda x: (x >= 0.7).mean()), s_par_appel=("secondes", "mean"))
              .sort_values("iou_moyen", ascending=False).round(3))
        DETECTION[form] = {"detail": D_, "tableau": T_}
        print(f"\n{form} : chaque modele cherche lui-meme le cadre de {len(qs)} questions sur "
              f"{min(N_SCANS_DETECTION, len(SCANS_PAR_FORM[form]))} scans (reference : le recalage sur le PDF)")
        display(T_)
        best = T_.index[0]
        if T_.loc[best, "iou_moyen"] >= 0.67:
            MEILLEUR_MODELE[form] = best
            print(f"-> secours pour les pages non recalees : {best} (IoU moyen {T_.loc[best, 'iou_moyen']:.2f})")
        else:
            print(f"-> aucun modele n'atteint un IoU moyen de 0,7 : pas de secours par modele "
                  f"(une page non recalee sera prise aux proportions et signalee dans l'index)")


In [ ]:
# 11. DECOUPE DES SCANS, QUESTION PAR QUESTION, deposee dans data/decoupes/<CAMPAGNE>/<FORM>/<bloc>/<scan>.jpg
#     + un INDEX (CSV, meme dossier) : pour chaque scan et chaque bloc, le chemin de la decoupe, la methode
#       (recalage / modele de secours / proportions) et la qualite du recalage. Les autres notebooks partent de l'index.
#     Reprise automatique : un scan deja dans l'index n'est pas refait (REFAIRE = True pour tout refaire, ex. apres
#     un nouveau gabarit). Arret propre apres DUREE_MAX_MINUTES : relancer la cellule continue.
REFAIRE = False
INDEX = {}


def chemin_index(form):
    return f"{DECOUPES}/{CAMPAGNE}/{form}/index_{form}_{CAMPAGNE}.csv"


def decoupe_par_modele(modele, img, bornes, b):
    """Secours : le modele cherche le cadre de la question sur la page (non recalee)."""
    a_, b_ = bornes[b["unite"] - 1]
    boite, _ = detecter(modele, img[a_:b_], b)
    if not boite:
        return None
    m_ = 0.01 * img.shape[1]
    x0, y0, x1, y1 = boite
    return img[max(0, int(y0 + a_ - m_)):int(y1 + a_ + m_), max(0, int(x0 - m_)):int(x1 + m_)]


def ecrire(a, chemin):
    b_ = io.BytesIO()
    if FORMAT == "png":
        Image.fromarray(a).save(b_, format="PNG", optimize=True)
    else:
        Image.fromarray(a).save(b_, format="JPEG", quality=92)
    with fs.open(chemin, "wb") as f_:
        f_.write(b_.getvalue())
    return len(b_.getvalue())


for form, scans_ in SCANS_PAR_FORM.items():
    nom = gabarit_pour(form)
    cfg = GABS[nom]["cfg"]
    R_ = G.Recaleur(cfg, local_)
    BLOCS_D = G.blocs_a_decouper(cfg, QUESTIONS)
    f_idx = chemin_index(form)
    deja = pd.read_csv(fs.open(f_idx)) if fs.exists(f_idx) and not REFAIRE else pd.DataFrame(columns=["scan", "gabarit"])
    faits = set(deja[deja.gabarit == nom].scan) if len(deja) else set()
    a_faire = [s for s in (scans_[:N_TEST] if TEST_RAPIDE else scans_) if s not in faits]
    print(f"{form} ({nom}) : {len(BLOCS_D)} blocs par scan | {len(scans_)} scans, {len(faits)} deja decoupes, "
          f"{len(a_faire)} a faire")
    STOP = threading.Event()
    t_debut = time.time()
    lignes_, verrou = [], threading.Lock()

    def un_scan(chemin):
        if STOP.is_set():
            return []
        img = lire_image(chemin)
        toile_, infos_ = R_.toile(img)
        bornes = R_.bornes_unites(img) if any(not i["recalee"] for i in infos_) else None
        out = []
        for b in BLOCS_D:
            inf = infos_[b["unite"] - 1]
            methode, crop = "recalage", None
            if not inf["recalee"] and SECOURS_PAR_MODELE and form in MEILLEUR_MODELE:
                crop = decoupe_par_modele(MEILLEUR_MODELE[form], img, bornes, b)
                methode = f"modele:{MEILLEUR_MODELE[form]}" if crop is not None and crop.size else "proportions"
            if crop is None or not crop.size:
                crop = R_.decouper(toile_, b["zone"], b.get("marge", 0.004))
                methode = "recalage" if inf["recalee"] else "proportions"
            dest = f"{DECOUPES}/{CAMPAGNE}/{form}/{b['id']}/{Path(chemin).stem}.{FORMAT}"
            taille = ecrire(crop, dest)
            out.append({"scan": chemin, "gabarit": nom, "bloc": b["id"], "question": b["question"], "theme": b.get("theme"),
                        "variables": " ".join(b["variables_formulaire"]), "decoupe": dest, "methode": methode,
                        "points_recalage": inf.get("retenus"), "angle": inf.get("angle"), "octets": taille})
        return out

    def sauver():
        with verrou:
            I_ = pd.concat([deja[~deja.scan.isin({l["scan"] for l in lignes_})] if len(deja) else deja,
                            pd.DataFrame(lignes_)], ignore_index=True)
        with fs.open(f_idx, "w") as f_:
            I_.to_csv(f_, index=False)
        return I_

    ex = ThreadPoolExecutor(N_PARALLELE_DECOUPE)
    futurs = [ex.submit(un_scan, s) for s in a_faire]
    try:
        for k_, fu in enumerate(as_completed(futurs), 1):
            r_ = fu.result()
            with verrou:
                lignes_ += r_
            if k_ % 50 == 0 or k_ == len(futurs):
                ecoule = time.time() - t_debut
                print(f"  {k_}/{len(futurs)} scans en {ecoule / 60:.1f} min "
                      f"(reste ~{ecoule / k_ * (len(futurs) - k_) / 60:.0f} min)", flush=True)
                sauver()
            if time.time() - t_debut > DUREE_MAX_MINUTES * 60:
                print(f"  DUREE_MAX_MINUTES atteinte : arret propre ({k_} scans faits). Relancer la cellule pour continuer.")
                break
    finally:
        STOP.set()
        ex.shutdown(wait=False, cancel_futures=True)
        INDEX[form] = sauver() if lignes_ else deja
    I_ = INDEX[form]
    if len(I_):
        par_methode = I_.drop_duplicates(["scan", "bloc"]).methode.value_counts(normalize=True).mul(100).round(1).to_dict()
        print(f"{form} : index {f_idx} | {I_.scan.nunique()} scans x {I_.bloc.nunique()} blocs | methode (%) : {par_methode}")
        if lignes_ and TEST_RAPIDE:
            moy = pd.DataFrame(lignes_).octets.mean()
            n_ = len({l["scan"] for l in lignes_})
            print(f"  volume prevu pour les {len(scans_)} scans : ~{moy * len(BLOCS_D) * len(scans_) / 1e9:.1f} Go ({FORMAT}) "
                  f"et ~{(time.time() - t_debut) / max(1, n_) * len(scans_) / 60:.0f} min")


In [ ]:
# 12. CONTROLE DES DECOUPES DE VRAIS SCANS
#     a) gemma4 relit un echantillon de decoupes : la question est-elle entiere, et seule ?
#     b) une mosaique (6 scans x 4 questions) pour l'oeil : HORS Git (vraies reponses), et MLflow interne.
CONSIGNE_VUE_SCAN = """Cette image est un morceau d'un bulletin du recensement rempli a la main.
Elle devrait montrer la question {num} « {intitule} ».
Reponds uniquement en JSON :
{{"numero_visible": "<le numero de question visible en haut a gauche, ou VIDE>",
  "bloc_entier": <true si la question est entiere, false si elle est coupee>,
  "autre_question_visible": <true si une AUTRE question apparait aussi>}}"""
CONTROLE_SCANS, MOSAIQUES = {}, {}
for form, I_ in INDEX.items():
    if not len(I_):
        continue
    nom = gabarit_pour(form)
    blocs_ = {b["id"]: b for b in GABS[nom]["cfg"]["blocs"]}
    ech = I_[I_.bloc.str.startswith("q")].sample(min(N_VERIF_DECOUPES, len(I_)), random_state=1) if N_VERIF_DECOUPES else I_.head(0)

    def relire(r):
        a = np.array(Image.open(io.BytesIO(fs.open(r.decoupe, "rb").read())).convert("RGB"))
        b = blocs_[r.bloc]
        v = demander_gemma([{"type": "text", "text": CONSIGNE_VUE_SCAN.format(num=b["question"], intitule=b["intitule"][:120])},
                            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + en_base64(a)}}])
        nv = re.sub(r"\D", "", str((v or {}).get("numero_visible", "")))
        return {"bloc": r.bloc, "reponse": isinstance(v, dict), "entiere": bool(v and v.get("bloc_entier") is True),
                "seule": bool(v and v.get("autre_question_visible") is False), "bon_numero": nv == str(b["question"])}

    with ThreadPoolExecutor(N_PARALLELE) as ex:
        C_ = pd.DataFrame(list(ex.map(relire, ech.itertuples())))
    CACHE_MODELE.write_text(json.dumps(CACHE_M, ensure_ascii=False))
    if len(C_):
        CONTROLE_SCANS[form] = {k: round(100 * float(C_[k].mean()), 1) for k in ["entiere", "seule", "bon_numero"]}
        print(f"{form} : sur {len(C_)} decoupes relues par {MODELE_GEMMA} -> question entiere {CONTROLE_SCANS[form]['entiere']} %, "
              f"seule {CONTROLE_SCANS[form]['seule']} %, bon numero lu {CONTROLE_SCANS[form]['bon_numero']} %")
        mauvaises = C_[~(C_.entiere & C_.seule & C_.bon_numero)].bloc.value_counts()
        if len(mauvaises):
            print(f"  a regarder : {mauvaises.to_dict()}")
    # mosaique
    qs_ = [q for q in ["q2", "q13", "q23", "q27"] if q in set(I_.bloc)] or list(I_.bloc.unique()[:4])
    scans_m = list(I_.scan.unique()[:6])
    hh, lignes_im = 150, []
    for s_ in scans_m:
        ims = []
        for q in qs_:
            r_ = I_[(I_.scan == s_) & (I_.bloc == q)]
            if len(r_):
                im = Image.open(io.BytesIO(fs.open(r_.decoupe.iloc[0], "rb").read())).convert("RGB")
                ims.append(im.resize((max(1, int(im.width * hh / im.height)), hh)))
        if ims:
            l_ = Image.new("RGB", (sum(i.width + 8 for i in ims), hh), "white")
            x0 = 0
            for im in ims:
                l_.paste(im, (x0, 0))
                x0 += im.width + 8
            lignes_im.append(l_)
    if lignes_im:
        mos = Image.new("RGB", (max(l.width for l in lignes_im), len(lignes_im) * (hh + 8)), "white")
        for k_, l_ in enumerate(lignes_im):
            mos.paste(l_, (0, k_ * (hh + 8)))
        MOSAIQUES[form] = ILLUS / f"mosaique_decoupes_{form}_{CAMPAGNE}.png"
        mos.save(MOSAIQUES[form])
        print(f"  mosaique ({len(lignes_im)} scans x {qs_}) : {MOSAIQUES[form]} (hors Git)")
        display(mos.resize((min(1200, mos.width), int(mos.height * min(1200, mos.width) / mos.width))))


## Utiliser ce travail dans un autre notebook

Tout est dans l'espace du projet, sous `data/decoupes/` :

- `gabarits/` : les configurations (`BI_2025.json`…), les pages vierges, l'image de contrôle, le catalogue Excel et le code commun `gabarits.py`.
- `<CAMPAGNE>/<FORM>/<bloc>/` : les découpes, **une par scan**. Exemple : `RG25/BI/q23/6901_5300000012_BI.jpg`.
- `<CAMPAGNE>/<FORM>/index_<FORM>_<CAMPAGNE>.csv` : l'index (scan, bloc, question, variables, chemin de la découpe, méthode).

**1) Lire les découpes déjà faites**, sans rien recalculer :

```python
import pandas as pd, s3fs
fs = s3fs.S3FileSystem()
idx = pd.read_csv(fs.open("projet-mesure-qualite-rp/data/decoupes/RG25/BI/index_BI_RG25.csv"))
q23 = idx[idx.variables.str.contains("NUMVOI", na=False)]     # par variable : stable d'un millesime a l'autre
# q23.decoupe = chemins des images ; q23.methode = "recalage" (sur) ou "proportions" (page non recalee, a ecarter)
```

**2) Découper autre chose** qu'un bloc entier, par exemple seulement les lignes de l'adresse avec l'intitulé :

```python
import sys, s3fs; fs = s3fs.S3FileSystem()
GAB = "projet-mesure-qualite-rp/data/decoupes/gabarits"
fs.get(f"{GAB}/gabarits.py", "/tmp/gabarits/gabarits.py"); sys.path.insert(0, "/tmp/gabarits")
import gabarits as G
cfg, R = G.charger(fs, GAB, "BI_2025")
sel = G.selectionner(cfg, variables=["NUMVOI", "CPLADR"], avec_intitule=True)   # ou libelle="Adresse", question=23...
toile, infos = R.toile(image_du_scan)              # tableau RGB du scan entier
decoupe = R.decouper(toile, sel["zone"], sel["marge"])
```

Le notebook 13 fait exactement cela : son `GABARIT` pointe vers `data/decoupes/gabarits/BI_2025.json`.


In [ ]:
# 14. MLflow : serveur du PROJET (identifiants dans Vault) ET serveur PERSONNEL. Un run par formulaire (gabarit,
#     tests, comparaison des modeles de detection, decoupe des scans) + le CODE du projet : gabarits.py (code commun)
#     et ce notebook sans ses sorties. Un serveur injoignable ne bloque pas l'autre.
import contextlib

SERVEURS_MLFLOW = ["projet", "personnel"]      # ["personnel"] pour le serveur personnel seul
DEMANDER_SI_VAULT_ECHOUE = True                # Vault ne repond pas : demander les identifiants du projet (saisie masquee)
SECRET_PROJET = "onyxia-kv/projet-mesure-qualite-rp/MLFLOW_SECRET"

manquants_ = [v for v in ["ENREGISTRER_MLFLOW", "GABS", "FICHIERS", "TEST_FAUX", "f_code"] if v not in globals()]
if manquants_:
    raise RuntimeError(f"absents de cette session : {', '.join(manquants_)}. Selectionner cette cellule, "
                       "puis Run > Run All Above Selected Cell (caches et reprise : rien n'est refait inutilement).")
for nom_v, defaut_ in [("INDEX", {}), ("DETECTION", {}), ("MEILLEUR_MODELE", {}), ("CONTROLE_SCANS", {}),
                       ("MOSAIQUES", {}), ("EVOLUTIONS", {})]:
    globals().setdefault(nom_v, defaut_)          # cellules 10 a 12 sautees : on enregistre ce qui existe

VARS_MLFLOW = ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD",
               "MLFLOW_TRACKING_TOKEN", "MLFLOW_TRACKING_INSECURE_TLS")


def config_personnelle():
    """MLflow du service tel qu'au demarrage (meme si une cellule a change la connexion depuis)."""
    try:
        brut_env = open("/proc/1/environ", "rb").read().split(b"\0")
        env = dict(x.decode(errors="ignore").split("=", 1) for x in brut_env if b"=" in x)
    except OSError:
        env = dict(os.environ)
    return {k: env[k] for k in VARS_MLFLOW if k in env}


def config_projet():
    """Identifiants du MLflow du PROJET : Vault (HTTP), sinon saisie masquee. Renvoie (config, raison si echec)."""
    if globals().get("_CONFIG_PROJET_SAISIE"):
        return globals()["_CONFIG_PROJET_SAISIE"], ""
    adresse, jeton = os.environ.get("VAULT_ADDR"), os.environ.get("VAULT_TOKEN")
    raison = ""
    if not adresse or not jeton:
        raison = "VAULT_ADDR ou VAULT_TOKEN absent de l'environnement"
    else:
        montage, reste = SECRET_PROJET.split("/", 1)
        try:
            r = requests.get(f"{adresse.rstrip('/')}/v1/{montage}/data/{reste}",
                             headers={"X-Vault-Token": jeton}, timeout=30)
            if r.status_code == 200:
                d = r.json()["data"]["data"]
                cfg_ = {k: str(v) for k, v in d.items() if k in VARS_MLFLOW}
                if "MLFLOW_TRACKING_URI" in cfg_:
                    return cfg_, ""
                raison = f"secret Vault lu, mais sans MLFLOW_TRACKING_URI (cles : {sorted(d)})"
            elif r.status_code == 403:
                raison = "Vault refuse le jeton (403) : jeton revoque ou expire -> redemarrer le service"
            elif r.status_code == 404:
                raison = f"secret introuvable dans Vault (404) : {SECRET_PROJET}"
            else:
                raison = f"Vault repond HTTP {r.status_code}"
        except requests.RequestException as e:
            raison = f"Vault injoignable ({type(e).__name__})"
    if DEMANDER_SI_VAULT_ECHOUE:
        print(f"MLflow projet : {raison}.\n  Saisie des identifiants du MLflow du PROJET ; Entree vide pour passer.")
        uri = input("  MLFLOW_TRACKING_URI (ex. https://projet-mesure-qualite-rp-mlflow.user.lab.sspcloud.fr) : ").strip()
        if uri:
            cfg_ = {"MLFLOW_TRACKING_URI": uri}
            u_ = _gp.getpass("  MLFLOW_TRACKING_USERNAME : ").strip()
            p_ = _gp.getpass("  MLFLOW_TRACKING_PASSWORD : ").strip()
            if u_:
                cfg_["MLFLOW_TRACKING_USERNAME"] = u_
            if p_:
                cfg_["MLFLOW_TRACKING_PASSWORD"] = p_
            globals()["_CONFIG_PROJET_SAISIE"] = cfg_          # en memoire pour la session, jamais sur disque
            return cfg_, ""
    return {}, raison


@contextlib.contextmanager
def sur_serveur(config):
    """Le temps d'un enregistrement, utilise ce serveur ; la connexion d'origine est restauree apres."""
    import mlflow
    avant, avant_uri = {k: os.environ.get(k) for k in VARS_MLFLOW}, mlflow.get_tracking_uri()
    try:
        for k in VARS_MLFLOW:
            if k in config:
                os.environ[k] = config[k]
            else:
                os.environ.pop(k, None)
        mlflow.set_tracking_uri(config["MLFLOW_TRACKING_URI"])
        yield
    finally:
        for k, v in avant.items():
            os.environ.pop(k, None) if v is None else os.environ.__setitem__(k, v)
        mlflow.set_tracking_uri(avant_uri)


def code_du_projet():
    """Les fichiers de code a joindre au run : gabarits.py, et ce notebook SANS ses sorties (pas d'image de scan)."""
    fichiers = [Path(f_code)]
    racine_ = next((p_ for p_ in [Path.cwd(), *Path.cwd().parents] if (p_ / "pyproject.toml").exists()), Path.cwd())
    nbs = sorted(racine_.glob("notebooks/*gabarit_formulaire*.ipynb")) or sorted(Path.cwd().glob("*gabarit_formulaire*.ipynb"))
    if nbs:
        nb_ = json.loads(nbs[0].read_text(encoding="utf-8"))
        for c_ in nb_.get("cells", []):
            if c_.get("cell_type") == "code":
                c_["outputs"], c_["execution_count"] = [], None
        f_nb = TRAVAIL / nbs[0].name
        f_nb.write_text(json.dumps(nb_, ensure_ascii=False, indent=1), encoding="utf-8")
        fichiers.append(f_nb)
    try:
        commit_ = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
                                 cwd=racine_).stdout.strip() or "inconnu"
        modifie_ = "oui" if subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                                           cwd=racine_).stdout.strip() else "non"
    except Exception:
        commit_, modifie_ = "inconnu", "inconnu"
    return fichiers, commit_, modifie_


def enregistrer_gabarit(nom, g_, nom_serveur="personnel"):
    import mlflow
    cfg = g_["cfg"]
    form = cfg["formulaire"]
    with mlflow.start_run(run_name=f"gabarit-{nom}-{CAMPAGNE}") as run_:
        mlflow.set_tags({"etude": "gabarits-decoupe", "notebook": "17_gabarit_formulaire", "formulaire": form,
                         "millesime": str(cfg["millesime"]), "campagne": CAMPAGNE, "source": cfg["source"][:200],
                         "depot": f"{DECOUPES}/gabarits/{nom}.json"})
        mlflow.log_params({"dpi": DPI, "modele_sens": MODELE_GEMMA if UTILISER_MODELE else "regles",
                           "verification_image": VERIFIER_PAR_IMAGE, "version_code": VERSION_CODE,
                           "questions_decoupees": str(QUESTIONS)[:250], "format": FORMAT, "test_rapide": TEST_RAPIDE,
                           "modele_secours": MEILLEUR_MODELE.get(form, "aucun")})
        c_ = cfg["controle"]
        met = {"blocs": float(c_["n_blocs"]), "champs": float(c_["n_champs"]),
               "questions": float(len(c_["questions_trouvees"])),
               "questions_manquantes": float(len(c_["questions_manquantes"])),
               "blocs_valides": float(c_.get("blocs_valides", 0))}
        if c_.get("recalage_faux_scans_ecart_max_mm") is not None:
            met["recalage_faux_scans_ecart_max_mm"] = float(c_["recalage_faux_scans_ecart_max_mm"])
        I_ = INDEX.get(form)
        if I_ is not None and len(I_) and gabarit_pour(form) == nom:
            u_ = I_.drop_duplicates(["scan", "bloc"])
            met.update({"scans_decoupes": float(I_.scan.nunique()), "decoupes": float(len(u_)),
                        "decoupes_par_recalage_pct": float(100 * (u_.methode == "recalage").mean()),
                        "decoupes_par_modele_pct": float(100 * u_.methode.str.startswith("modele").mean()),
                        "decoupes_aux_proportions_pct": float(100 * (u_.methode == "proportions").mean())})
            met.update({f"controle_gemma_{k}_pct": float(v) for k, v in CONTROLE_SCANS.get(form, {}).items()})
            if form in MOSAIQUES:
                mlflow.log_artifact(str(MOSAIQUES[form]), "decoupes_vrais_scans")
        if form in DETECTION and gabarit_pour(form) == nom:
            T_ = DETECTION[form]["tableau"]
            for m_, r_ in T_.iterrows():
                k_ = re.sub(r"[^A-Za-z0-9_.-]", "_", m_)[:50]
                met[f"detection_{k_}_iou_moyen"] = float(r_.iou_moyen)
                met[f"detection_{k_}_s_par_appel"] = float(r_.s_par_appel)
            f_d = TRAVAIL / f"detection_modeles_{form}_{CAMPAGNE}.csv"
            DETECTION[form]["detail"].to_csv(f_d, index=False)
            mlflow.log_artifact(str(f_d), "detection_modeles")
        mlflow.log_metrics(met)
        for k_ in ["json", "controle", "catalogue"]:
            mlflow.log_artifact(str(FICHIERS[nom][k_]), "gabarit")
        f_t = TRAVAIL / f"test_recalage_faux_scans_{nom}.csv"
        TEST_FAUX[TEST_FAUX.gabarit == nom].to_csv(f_t, index=False)
        mlflow.log_artifact(str(f_t), "tests")
        if nom in EVOLUTIONS and len(EVOLUTIONS[nom]):
            f_e = TRAVAIL / f"evolutions_{nom}.csv"
            EVOLUTIONS[nom].to_csv(f_e, index=False)
            mlflow.log_artifact(str(f_e), "tests")
        fichiers_code, commit_, modifie_ = code_du_projet()
        mlflow.set_tags({"git_commit": commit_, "git_code_modifie": modifie_, "serveur": nom_serveur})
        for f_ in fichiers_code:
            mlflow.log_artifact(str(f_), "code")
        presents = sorted(x.path for x in mlflow.artifacts.list_artifacts(run_id=run_.info.run_id))
    return run_.info.run_id[:8], presents


if ENREGISTRER_MLFLOW:
    import mlflow
    os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "30")
    os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "2")
    RESULTATS_MLFLOW = {}
    for nom_serveur in SERVEURS_MLFLOW:
        if nom_serveur == "projet":
            config_, raison_ = config_projet()
        else:
            config_ = config_personnelle()
            raison_ = "" if "MLFLOW_TRACKING_URI" in config_ else "MLflow non configure dans ce service"
        if "MLFLOW_TRACKING_URI" not in config_:
            RESULTATS_MLFLOW[nom_serveur] = f"non enregistre : {raison_}"
            print(f"MLflow {nom_serveur} : NON ENREGISTRE - {raison_}")
            continue
        hote_ = config_["MLFLOW_TRACKING_URI"].split("//")[-1].split("/")[0]
        for nom, g_ in GABS.items():
            try:
                with sur_serveur(config_):
                    mlflow.set_experiment("fiqual-banc-essai")
                    run_court, presents = enregistrer_gabarit(nom, g_, nom_serveur)
                RESULTATS_MLFLOW[f"{nom_serveur} {nom}"] = run_court
                print(f"MLflow {nom_serveur} ({hote_}) : {nom} enregistre, essai {run_court} | dossiers : {', '.join(presents)}")
            except Exception as e:
                RESULTATS_MLFLOW[f"{nom_serveur} {nom}"] = f"echec : {type(e).__name__}"
                print(f"MLflow {nom_serveur} ({hote_}) : echec pour {nom} - {type(e).__name__} : {str(e)[:150]}")
    print(RESULTATS_MLFLOW)
else:
    print("ENREGISTRER_MLFLOW = False (cellule 1) : rien n'est enregistre")